# Reversible Circuits Project: Phase 11

Two extensions of the project.

**B. Rewrite rules for ancilla and garbage lines.** The complete rule set of Feng and Li covers circuits without ancillary bits only. `arules.py` adds five local rules for circuits whose ancilla lines start at a constant and whose garbage lines are never read:

| Rule | What it does |
|---|---|
| A1 | A control on an ancilla line that no gate has written yet is always or never satisfied: drop the control, or delete the gate |
| A2 | A gate whose target is not an output and is never read again is deleted |
| A3 | After a fresh ancilla c is set to a product of literals, c = 1 implies those literals while their lines are untouched: contradicting gates are deleted, implied controls dropped |
| A4 | If c equals one literal, a control on c is replaced by that literal |
| A5 | If c was built from 0 by CNOT gates only, a CNOT controlled by c is replaced by CNOTs from the source lines |

Each rule looks only at the gates before (or after) the gate it rewrites; no simulation of the whole circuit is needed. Part B of the script compares them with the global simplifier of Phase 5.

**A. A hybrid carry look-ahead adder.** `hybrid.py` keeps the Peres-based PGA blocks of Kumar et al., folds the carry input into the first generate bit, and computes all carries with the logarithmic-depth prefix network of Draper et al., without uncomputing (garbage style, like Kumar et al.).

**How to run:** choose **Runtime > Run all**. It takes about two minutes. Expected last line: `All checks passed.`

## Step 1: Project files

In [ ]:
%%writefile revcircuit.py
"""Reversible circuit model and simulator (Phase 1).

Gates are mixed-polarity multiple-control Toffoli (MPMCT) gates, written
G[P, N, q] as in Feng and Li, "A Complete Set of Transformation Rules for
Reversible Circuits", IEEE TCAD, 2026:

    P  set of positive control lines
    N  set of negative control lines
    q  target line

The gate flips q when every line in P is 1 and every line in N is 0.
Special cases: X[q] = G[{}, {}, q], CNOT[p, q] = G[{p}, {}, q], and a
Toffoli gate is G[{a, b}, {}, q].

A basis state is stored as an integer whose bit i is the value of line q_i.
A circuit is a list of gates; the leftmost gate executes first.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple


# ---------------------------------------------------------------------------
# Gate
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Gate:
    """One MPMCT gate G[pos, neg, target]."""

    pos: frozenset
    neg: frozenset
    target: int

    def __post_init__(self) -> None:
        object.__setattr__(self, "pos", frozenset(self.pos))
        object.__setattr__(self, "neg", frozenset(self.neg))
        if self.pos & self.neg:
            raise ValueError("a line cannot be both a positive and a negative control")
        if self.target in self.pos or self.target in self.neg:
            raise ValueError("the target line cannot also be a control")
        if min(self.lines) < 0:
            raise ValueError("line numbers must be non-negative")
        # Bit masks used by the simulator.
        object.__setattr__(self, "_pmask", sum(1 << l for l in self.pos))
        object.__setattr__(self, "_nmask", sum(1 << l for l in self.neg))
        object.__setattr__(self, "_tmask", 1 << self.target)

    @property
    def controls(self) -> frozenset:
        return self.pos | self.neg

    @property
    def lines(self) -> frozenset:
        return self.pos | self.neg | {self.target}

    def apply(self, state: int) -> int:
        """Return the state after this gate acts on `state`."""
        if (state & self._pmask) == self._pmask and not (state & self._nmask):
            return state ^ self._tmask
        return state

    def __str__(self) -> str:
        t = self.target
        if not self.neg:
            if not self.pos:
                return f"X[q{t}]"
            if len(self.pos) == 1:
                (c,) = self.pos
                return f"CNOT[q{c},q{t}]"

        def fmt(s: frozenset) -> str:
            return "{" + ",".join(f"q{l}" for l in sorted(s)) + "}"

        return f"G[{fmt(self.pos)},{fmt(self.neg)},q{t}]"


def G(pos: Iterable[int], neg: Iterable[int], target: int) -> Gate:
    """General gate G[P, N, q]."""
    return Gate(frozenset(pos), frozenset(neg), target)


def X(q: int) -> Gate:
    """NOT gate on line q."""
    return Gate(frozenset(), frozenset(), q)


def CNOT(control: int, target: int) -> Gate:
    """Controlled-NOT (Feynman gate)."""
    return Gate(frozenset({control}), frozenset(), target)


def TOF(c1: int, c2: int, target: int) -> Gate:
    """Toffoli gate with two positive controls."""
    return Gate(frozenset({c1, c2}), frozenset(), target)


def MCT(controls: Iterable[int], target: int) -> Gate:
    """Multiple-control Toffoli gate with positive controls only."""
    return Gate(frozenset(controls), frozenset(), target)


# ---------------------------------------------------------------------------
# Circuit
# ---------------------------------------------------------------------------
class Circuit:
    """A reversible circuit on n lines: an ordered list of gates."""

    def __init__(self, n: int, gates: Iterable[Gate] = ()) -> None:
        self.n = n
        self.gates: List[Gate] = list(gates)
        for g in self.gates:
            if max(g.lines) >= n:
                raise ValueError(f"gate {g} uses a line outside q0..q{n - 1}")

    # -- container behaviour ------------------------------------------------
    def __len__(self) -> int:
        return len(self.gates)

    def __getitem__(self, i: int) -> Gate:
        return self.gates[i]

    def __iter__(self) -> Iterator[Gate]:
        return iter(self.gates)

    def __eq__(self, other: object) -> bool:
        """Structural equality: same lines and the same gate list."""
        return isinstance(other, Circuit) and self.n == other.n and self.gates == other.gates

    def __add__(self, other: "Circuit") -> "Circuit":
        """Concatenation: self executes first, then other."""
        if self.n != other.n:
            raise ValueError("circuits must have the same number of lines")
        return Circuit(self.n, self.gates + other.gates)

    def __str__(self) -> str:
        return " ".join(str(g) for g in self.gates) if self.gates else "(empty)"

    def copy(self) -> "Circuit":
        return Circuit(self.n, self.gates)

    def replace(self, i: int, j: int, new_gates: Sequence[Gate]) -> "Circuit":
        """Return a new circuit with gates[i:j] replaced by new_gates."""
        return Circuit(self.n, self.gates[:i] + list(new_gates) + self.gates[j:])

    def inverse(self) -> "Circuit":
        """Every gate is its own inverse, so the inverse is the reversed list."""
        return Circuit(self.n, reversed(self.gates))

    # -- simulation ---------------------------------------------------------
    def apply(self, state: int) -> int:
        for g in self.gates:
            state = g.apply(state)
        return state

    def permutation(self) -> Tuple[int, ...]:
        """The reversible function computed, as a tuple over all 2^n states."""
        return tuple(self.apply(s) for s in range(1 << self.n))

    def equivalent(self, other: "Circuit") -> bool:
        """Functional equivalence: both circuits compute the same function."""
        return self.n == other.n and self.permutation() == other.permutation()

    # -- display ------------------------------------------------------------
    def _bits(self, state: int) -> str:
        return " ".join(str((state >> i) & 1) for i in range(self.n))

    def truth_table(self) -> str:
        head = " ".join(f"q{i}" for i in range(self.n))
        rows = [f"{head} | {head}"]
        for s, t in enumerate(self.permutation()):
            pad = "  ".join(self._bits(s).split())
            out = "  ".join(self._bits(t).split())
            rows.append(f" {pad} |  {out}")
        return "\n".join(rows)

    def draw(self) -> str:
        """Text drawing: circled plus = target, filled dot = positive control, open dot = negative control."""
        width = len(f"q{self.n - 1}")
        rows = []
        for r in range(self.n):
            cells = []
            for g in self.gates:
                lo, hi = min(g.lines), max(g.lines)
                if r == g.target:
                    ch = "\u2295"  # target
                elif r in g.pos:
                    ch = "\u25cf"  # filled dot
                elif r in g.neg:
                    ch = "\u25cb"  # open dot
                elif lo < r < hi:
                    ch = "\u253c"  # wire crossing a gate
                else:
                    ch = "\u2500"
                cells.append("\u2500" + ch + "\u2500")
            rows.append(f"q{r}:".ljust(width + 2) + "\u2500" + "".join(cells) + "\u2500")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# Helpers for experiments
# ---------------------------------------------------------------------------
def pn_choices(lines: Sequence[int]) -> Iterator[Tuple[frozenset, frozenset]]:
    """All ways to make each given line absent, a positive or a negative control."""
    for states in product((0, 1, 2), repeat=len(lines)):
        pos = frozenset(l for l, s in zip(lines, states) if s == 1)
        neg = frozenset(l for l, s in zip(lines, states) if s == 2)
        yield pos, neg


def all_gates(n: int) -> List[Gate]:
    """Every MPMCT gate on n lines: n * 3^(n-1) gates."""
    gates = []
    for target in range(n):
        others = [l for l in range(n) if l != target]
        for pos, neg in pn_choices(others):
            gates.append(Gate(pos, neg, target))
    return gates


def random_gate(n: int, rng: Optional[random.Random] = None) -> Gate:
    rng = rng or random
    target = rng.randrange(n)
    pos, neg = set(), set()
    for l in range(n):
        if l == target:
            continue
        s = rng.randrange(3)
        if s == 1:
            pos.add(l)
        elif s == 2:
            neg.add(l)
    return Gate(frozenset(pos), frozenset(neg), target)


def random_circuit(n: int, length: int, rng: Optional[random.Random] = None) -> Circuit:
    return Circuit(n, [random_gate(n, rng) for _ in range(length)])

In [ ]:
%%writefile rules.py
"""Transformation rules for reversible circuits (Phase 1).

Rules 1 to 5 are the complete rule set RC of Feng and Li (IEEE TCAD, 2026).
Rules 6 to 10 are the rules the paper derives from RC.

Every rewrite function takes a circuit and a position i and returns a NEW
circuit with the rule applied at position i, or None when the rule does not
match there. The input circuit is never modified.

For rules stated on gates rather than positions, a `ruleK_gates` or
`ruleK_sides` helper returns the gate lists on each side of the rule.
"""
from __future__ import annotations

from itertools import combinations
from typing import Iterable, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X


# ===========================================================================
# Rule 1:  A A  ==  (empty)
# ===========================================================================
def rule1_cancel(c: Circuit, i: int) -> Optional[Circuit]:
    """Delete two adjacent identical gates at positions i and i+1."""
    if 0 <= i and i + 1 < len(c) and c[i] == c[i + 1]:
        return c.replace(i, i + 2, [])
    return None


def rule1_insert(c: Circuit, i: int, gate: Gate) -> Circuit:
    """Rule 1 read right to left: insert the pair `gate gate` before position i."""
    return c.replace(i, i, [gate, gate])


# ===========================================================================
# Rule 2:  G[P, N+{p}, q] G[P+{p}, N, q]  ==  G[P, N, q]
# ===========================================================================
def rule2_gates(gate: Gate, p: int) -> List[Gate]:
    """The pair A0, A1 that Rule 2 merges into `gate`, using extra line p."""
    if p in gate.lines:
        raise ValueError("p must be a line the gate does not use")
    return [G(gate.pos, gate.neg | {p}, gate.target), G(gate.pos | {p}, gate.neg, gate.target)]


def rule2_merge(c: Circuit, i: int) -> Optional[Circuit]:
    """Merge gates i and i+1 when they are A0 A1 in the order of the paper."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    extra = a1.pos - a0.pos
    if a0.target != a1.target or len(extra) != 1:
        return None
    (p,) = extra
    merged = G(a1.pos - {p}, a1.neg, a1.target)
    if rule2_gates(merged, p) == [a0, a1]:
        return c.replace(i, i + 2, [merged])
    return None


def rule2_split(c: Circuit, i: int, p: int) -> Optional[Circuit]:
    """Rule 2 read right to left: split gate i into A0 A1 on unused line p."""
    if not (0 <= i < len(c)) or not (0 <= p < c.n) or p in c[i].lines:
        return None
    return c.replace(i, i + 1, rule2_gates(c[i], p))


# ===========================================================================
# Rule 3:  A B == B A  when a control has opposite polarity in A and B
# ===========================================================================
def rule3_condition(a: Gate, b: Gate) -> bool:
    return bool(a.pos & b.neg) or bool(b.pos & a.neg)


def rule3_commute(c: Circuit, i: int) -> Optional[Circuit]:
    """Swap gates i and i+1 when they share a control of opposite polarity."""
    if 0 <= i and i + 1 < len(c) and rule3_condition(c[i], c[i + 1]):
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# ===========================================================================
# Rule 4:  A B A C1 == C2 A B A,  A = CNOT[p,q], B = CNOT[q,p]
#          (moving a swap of p and q through a gate)
# ===========================================================================
def rule4_sides(p: int, q: int, P: Iterable[int], N: Iterable[int],
                negative: bool = False) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 4.

    negative=False is condition (a): C1 has q as positive control, target p.
    negative=True  is condition (b): C1 has q as negative control, target p.
    """
    P, N = frozenset(P), frozenset(N)
    a, b = CNOT(p, q), CNOT(q, p)
    if negative:
        c1, c2 = G(P, N | {q}, p), G(P, N | {p}, q)
    else:
        c1, c2 = G(P | {q}, N, p), G(P | {p}, N, q)
    return [a, b, a, c1], [c2, a, b, a]


def _is_cnot(g: Gate) -> bool:
    return len(g.pos) == 1 and not g.neg


def rule4_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Apply Rule 4 to the four gates starting at i, in either direction."""
    if i < 0 or i + 4 > len(c):
        return None
    seg = c.gates[i:i + 4]
    # Left to right:  A B A C1  ->  C2 A B A
    if _is_cnot(seg[0]):
        (p,) = seg[0].pos
        q = seg[0].target
        c1 = seg[3]
        if c1.target == p and q in c1.controls:
            lhs, rhs = rule4_sides(p, q, c1.pos - {q}, c1.neg - {q}, q in c1.neg)
            if seg == lhs:
                return c.replace(i, i + 4, rhs)
    # Right to left:  C2 A B A  ->  A B A C1
    if _is_cnot(seg[1]):
        (p,) = seg[1].pos
        q = seg[1].target
        c2 = seg[0]
        if c2.target == q and p in c2.controls:
            lhs, rhs = rule4_sides(p, q, c2.pos - {p}, c2.neg - {p}, p in c2.neg)
            if seg == rhs:
                return c.replace(i, i + 4, lhs)
    return None


# ===========================================================================
# Rule 5:  A0 A1 B1..Bm..B1 A1 A0  ==  B'1..B'm..B'1
#          (changes the polarity of control q0)
# ===========================================================================
def rule5_sides(P: Iterable[int], N: Iterable[int], Q: Sequence[int],
                q0: int) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 5 for control sets P, N, ordered list Q and line q0."""
    P, N, Q = frozenset(P), frozenset(N), list(Q)
    Qs = frozenset(Q)
    if not Q or len(Qs) != len(Q):
        raise ValueError("Q must be a non-empty list of distinct lines")
    a0 = G(P, N | Qs, q0)
    a1 = G(P | Qs, N, q0)
    b = [G(P | {q0} | frozenset(Q[k + 1:]), N | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    bp = [G(P | frozenset(Q[k + 1:]), N | {q0} | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    lhs = [a0, a1] + b + b[-2::-1] + [a1, a0]
    rhs = bp + bp[-2::-1]
    return lhs, rhs


def rule5_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Left to right: replace the 2m+3 gate pattern starting at i by 2m-1 gates."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    Qs = a1.pos - a0.pos
    if a0.target != a1.target or not Qs or a0.neg - a1.neg != Qs:
        return None
    if a0.pos - a1.pos or a1.neg - a0.neg:
        return None
    m = len(Qs)
    end = i + 2 * m + 3
    if end > len(c):
        return None
    order = [g.target for g in c.gates[i + 2:i + 2 + m]]
    if frozenset(order) != Qs:
        return None
    lhs, rhs = rule5_sides(a0.pos, a1.neg, order, a0.target)
    if c.gates[i:end] == lhs:
        return c.replace(i, end, rhs)
    return None


def rule5_reverse(c: Circuit, i: int, Q: Sequence[int], q0: int) -> Optional[Circuit]:
    """Right to left: expand B'1..B'm..B'1 at i, given the list Q and line q0."""
    Q = list(Q)
    end = i + 2 * len(Q) - 1
    if i < 0 or not Q or end > len(c):
        return None
    b1 = c[i]
    if b1.target != Q[0] or q0 not in b1.neg:
        return None
    try:
        lhs, rhs = rule5_sides(b1.pos - frozenset(Q[1:]), b1.neg - {q0}, Q, q0)
    except ValueError:
        return None
    if c.gates[i:end] == rhs:
        return c.replace(i, end, lhs)
    return None


# ===========================================================================
# Derived rules 6 to 10
# ===========================================================================
# Rule 6:  G[P,N,q] == product over all subsets S of Q of G[P+S, N+(Q-S), q]
def rule6_gates(gate: Gate, Q: Iterable[int]) -> List[Gate]:
    Q = sorted(set(Q))
    if set(Q) & gate.lines:
        raise ValueError("Q must not overlap the lines of the gate")
    out = []
    for r in range(len(Q) + 1):
        for sub in combinations(Q, r):
            out.append(G(gate.pos | set(sub), gate.neg | (set(Q) - set(sub)), gate.target))
    return out


def rule6_expand(c: Circuit, i: int, Q: Iterable[int]) -> Optional[Circuit]:
    """Expand gate i over all polarity patterns of the unused lines Q."""
    Q = set(Q)
    if not (0 <= i < len(c)) or not Q or Q & c[i].lines or max(Q) >= c.n or min(Q) < 0:
        return None
    return c.replace(i, i + 1, rule6_gates(c[i], Q))


# Rule 7:  two gates with the same target commute
def rule7_commute(c: Circuit, i: int) -> Optional[Circuit]:
    if 0 <= i and i + 1 < len(c) and c[i].target == c[i + 1].target:
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# Rule 8:  G[P,N,q] == X[N] G[P+N, {}, q] X[N]
def rule8_gates(gate: Gate) -> List[Gate]:
    xs = [X(l) for l in sorted(gate.neg)]
    return xs + [G(gate.pos | gate.neg, (), gate.target)] + xs


def rule8_expand(c: Circuit, i: int) -> Optional[Circuit]:
    """Replace the negative controls of gate i by X gates before and after."""
    if 0 <= i < len(c) and c[i].neg:
        return c.replace(i, i + 1, rule8_gates(c[i]))
    return None


def rule8_collapse(c: Circuit, i: int) -> Optional[Circuit]:
    """Right to left: absorb X..X B X..X starting at i into one gate."""
    k = 0
    while i + k < len(c) and not c[i + k].controls:
        k += 1
    if i < 0 or k == 0 or i + 2 * k + 1 > len(c):
        return None
    b = c[i + k]
    lines = frozenset(g.target for g in c.gates[i:i + k])
    if len(lines) != k or b.neg or not lines <= b.pos:
        return None
    merged = G(b.pos - lines, lines, b.target)
    if c.gates[i:i + 2 * k + 1] == rule8_gates(merged):
        return c.replace(i, i + 2 * k + 1, [merged])
    return None


# Rule 9:  A B1 == B2 A, where A targets p and B1, B2 differ in the polarity of p
def _rule9_pair(a: Gate, b: Gate) -> Optional[Gate]:
    """If A can pass B by Rule 9, return B with the polarity of A's target flipped."""
    p = a.target
    if p not in b.controls:
        return None
    if not (a.pos <= b.pos - {p} and a.neg <= b.neg - {p}):
        return None
    if p in b.pos:
        return G(b.pos - {p}, b.neg | {p}, b.target)
    return G(b.pos | {p}, b.neg - {p}, b.target)


def rule9_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Move a gate past its neighbour, flipping one control polarity in the neighbour."""
    if i < 0 or i + 1 >= len(c):
        return None
    g1, g2 = c[i], c[i + 1]
    flipped = _rule9_pair(g1, g2)          # A B1 -> B2 A
    if flipped is not None:
        return c.replace(i, i + 2, [flipped, g1])
    flipped = _rule9_pair(g2, g1)          # B2 A -> A B1
    if flipped is not None:
        return c.replace(i, i + 2, [g2, flipped])
    return None


# Rule 10:  A B A == B A B,  A = G[P+{p}, {}, q],  B = G[P+{q}, {}, p]
def rule10_sides(P: Iterable[int], p: int, q: int) -> Tuple[List[Gate], List[Gate]]:
    P = frozenset(P)
    a, b = G(P | {p}, (), q), G(P | {q}, (), p)
    return [a, b, a], [b, a, b]


def rule10_apply(c: Circuit, i: int) -> Optional[Circuit]:
    if i < 0 or i + 3 > len(c):
        return None
    a, b = c[i], c[i + 1]
    p, q = b.target, a.target
    if a.neg or b.neg or p == q or p not in a.pos or q not in b.pos:
        return None
    if a.pos - {p} != b.pos - {q}:
        return None
    lhs, rhs = rule10_sides(a.pos - {p}, p, q)
    if c.gates[i:i + 3] == lhs:
        return c.replace(i, i + 3, rhs)
    return None

In [ ]:
%%writefile optimizer.py
"""Heuristic rule-based optimizer for reversible circuits (Phase 3).

The optimizer rewrites a circuit with the transformation rules of Feng and
Li (IEEE TCAD, 2026) until the cost stops falling. One search step takes a
gate, moves it along the circuit, and looks for a partner it can cancel or
merge with.

Moves that remove gates
    Rule 1          A A                      ->  (nothing)
    Rule 2          G[P,N+p,q] G[P+p,N,q]    ->  G[P,N,q]
    Rule 2 + 1      G[P,N,q]   G[P+p,N,q]    ->  G[P,N+p,q]      ("absorb")
    Rule 5          the 2m+3 gate pattern    ->  2m-1 gates

Moves that carry a gate past its neighbour
    Rule 7          same target: the gates commute
    Rule 3          a shared control of opposite polarity: they commute
    D1 (derived)    neither target is a control of the other: they commute
    Rule 9          the gate passes; a control polarity flips in one gate
    D2 (derived)    the gate passes and leaves one extra gate behind

D1 and D2 are not listed in the paper. Both are sound (verify_phase3.py
checks every instance on 2 to 4 lines), so by the completeness theorem of
the paper they follow from Rules 1 to 5. Rule 9 is the special case of D2
in which the extra gate merges with its neighbour.

Every accepted rewrite strictly lowers the objective, so the search stops.
With check=True each rewrite is also verified by simulating the replaced
gates and the new gates on the lines they use.
"""
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Sequence, Tuple

from revcircuit import Circuit, G, Gate
import rules as R

CostFn = Callable[[Gate], int]


# ---------------------------------------------------------------------------
# Cost model
# ---------------------------------------------------------------------------
def quantum_cost(gate: Gate) -> int:
    """Quantum cost of one mixed-polarity multiple-control Toffoli gate.

    Controls:  0   1   2   3    4    5   ...   c
    Cost:      1   1   5   13   29   61  ...   2^(c+1) - 3

    This is the count of elementary gates in the decomposition of Barenco et
    al. (1995) without ancilla lines. Negative controls are free, except that
    a gate whose controls are ALL negative costs one more. The model is a
    stated assumption of this project; pass another function to change it.
    """
    c = len(gate.pos) + len(gate.neg)
    cost = 1 if c <= 1 else (1 << (c + 1)) - 3
    if c >= 1 and not gate.pos:
        cost += 1
    return cost


def circuit_cost(circuit, cost: CostFn = quantum_cost) -> int:
    """Total quantum cost of a circuit or a list of gates."""
    return sum(cost(g) for g in circuit)


# ---------------------------------------------------------------------------
# Elementary moves
# ---------------------------------------------------------------------------
def reduce_pair(a: Gate, b: Gate) -> Optional[Tuple[List[Gate], str]]:
    """Replace two gates with the same target by at most one gate, if possible."""
    if a.target != b.target:
        return None
    if a == b:
        return [], "Rule 1"
    la, lb = a.controls, b.controls
    if la == lb:
        differ = a.pos ^ b.pos                      # lines with opposite polarity
        if len(differ) == 1:
            (p,) = differ
            return [G(a.pos - {p}, a.neg - {p}, a.target)], "Rule 2"
    elif len(la ^ lb) == 1:
        big, small = (a, b) if len(la) > len(lb) else (b, a)
        (p,) = big.controls - small.controls
        if big.pos - {p} == small.pos and big.neg - {p} == small.neg:
            return [_flip(big, p)], "Rule 2 + 1"
    return None


def _flip(gate: Gate, p: int) -> Gate:
    """The same gate with the polarity of control p reversed."""
    if p in gate.pos:
        return G(gate.pos - {p}, gate.neg | {p}, gate.target)
    return G(gate.pos | {p}, gate.neg - {p}, gate.target)


def commute_tag(a: Gate, b: Gate) -> Optional[str]:
    """Name of the rule by which two adjacent gates commute, or None."""
    if a.target == b.target:
        return "Rule 7"
    if (a.pos & b.neg) or (a.neg & b.pos):
        return "Rule 3"
    if a.target not in b.controls and b.target not in a.controls:
        return "D1"
    return None


def induced_gate(a: Gate, b: Gate) -> Optional[Gate]:
    """The extra gate of rule D2.

    A = G[P1,N1,p] writes line p, and B = G[..p.., q] reads it. Then
        A B  ==  B B'' A      and      B A  ==  A B'' B
    with B'' = G[P1 + P2, N1 + N2, q], where P2, N2 are the controls of B
    other than p. Returns None when the rule does not apply.
    """
    p, q = a.target, b.target
    if p not in b.controls or q in a.controls:
        return None
    pos, neg = a.pos | (b.pos - {p}), a.neg | (b.neg - {p})
    if pos & neg:
        return None                                  # then Rule 3 applies instead
    return G(pos, neg, q)


def pass_gate(moving: Gate, fixed: Gate) -> Optional[Tuple[List[Gate], Gate, str, int]]:
    """Carry `moving` from the left of `fixed` to its right.

    Returns (gates left behind, the moving gate afterwards, rule, extra gates)
    or None when neither Rule 9 nor D2 applies.
    """
    extra = induced_gate(moving, fixed)              # moving writes, fixed reads
    if extra is not None:
        merged = reduce_pair(fixed, extra)
        if merged is not None:
            return merged[0], moving, "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    extra = induced_gate(fixed, moving)              # fixed writes, moving reads
    if extra is not None:
        merged = reduce_pair(extra, moving)
        if merged is not None:
            return [fixed], merged[0][0], "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    return None


# ---------------------------------------------------------------------------
# Checking one rewrite
# ---------------------------------------------------------------------------
def same_function(old: Sequence[Gate], new: Sequence[Gate],
                  rng: Optional[random.Random] = None, limit: int = 12) -> bool:
    """True when two gate lists act identically on the lines they use.

    Exhaustive for up to `limit` lines; beyond that, 4096 random states.
    """
    lines = sorted(set().union(*(g.lines for g in list(old) + list(new)))) if (old or new) else []
    index = {l: i for i, l in enumerate(lines)}

    def remap(gates):
        return [G([index[l] for l in g.pos], [index[l] for l in g.neg], index[g.target])
                for g in gates]

    a, b = remap(old), remap(new)
    k = len(lines)
    if k <= limit:
        states = range(1 << k)
    else:
        rng = rng or random.Random(0)
        states = [rng.getrandbits(k) for _ in range(4096)]
    for s in states:
        x = y = s
        for g in a:
            x = g.apply(x)
        for g in b:
            y = g.apply(y)
        if x != y:
            return False
    return True


# ---------------------------------------------------------------------------
# The optimizer
# ---------------------------------------------------------------------------
@dataclass
class OptimizationResult:
    original: Circuit
    circuit: Circuit
    rule_counts: Counter = field(default_factory=Counter)
    rewrites: int = 0
    passes: int = 0
    cost_fn: CostFn = quantum_cost

    @property
    def gates_before(self) -> int:
        return len(self.original)

    @property
    def gates_after(self) -> int:
        return len(self.circuit)

    @property
    def cost_before(self) -> int:
        return circuit_cost(self.original, self.cost_fn)

    @property
    def cost_after(self) -> int:
        return circuit_cost(self.circuit, self.cost_fn)

    def summary(self) -> str:
        def change(before, after):
            pct = 100.0 * (before - after) / before if before else 0.0
            return f"{before} -> {after} ({pct:.1f}% lower)"

        lines = [f"Gate count:   {change(self.gates_before, self.gates_after)}",
                 f"Quantum cost: {change(self.cost_before, self.cost_after)}",
                 f"Rewrites accepted: {self.rewrites} in {self.passes} passes"]
        if self.rule_counts:
            lines.append("Rule usage: " + ", ".join(
                f"{name} x{count}" for name, count in sorted(self.rule_counts.items())))
        return "\n".join(lines)


class _Search:
    def __init__(self, objective: str, window: int, extra_gates: int,
                 cost: CostFn, check: bool) -> None:
        if objective not in ("cost", "gates"):
            raise ValueError('objective must be "cost" or "gates"')
        self.objective, self.window, self.extra_gates = objective, window, extra_gates
        self.cost, self.check = cost, check
        self.counts: Counter = Counter()
        self.rewrites = 0
        self.rng = random.Random(0)

    def score(self, gates: Sequence[Gate]) -> Tuple[int, int]:
        qc, count = sum(self.cost(g) for g in gates), len(gates)
        return (qc, count) if self.objective == "cost" else (count, qc)

    def walk(self, gates: List[Gate], i: int):
        """Move gate i to the right; return the best improving rewrite found."""
        moving, left, tags, growth = gates[i], [], [], 0
        best = None
        for k in range(i + 1, min(len(gates), i + 1 + self.window)):
            fixed = gates[k]
            reduced = reduce_pair(moving, fixed)
            if reduced is not None:
                new = left + reduced[0]
                gain = tuple(o - n for o, n in zip(self.score(gates[i:k + 1]), self.score(new)))
                if gain > (0, 0) and (best is None or gain > best[0]):
                    best = (gain, k, new, tags + [reduced[1]])
            tag = commute_tag(moving, fixed)
            if tag is not None:
                left, tags = left + [fixed], tags + [tag]
                continue
            passed = pass_gate(moving, fixed)
            if passed is None or growth + passed[3] > self.extra_gates:
                break
            left, moving = left + passed[0], passed[1]
            tags, growth = tags + [passed[2]], growth + passed[3]
        return best

    def sweep(self, gates: List[Gate]) -> bool:
        """One left-to-right sweep. Returns True when the circuit changed."""
        changed, i = False, 0
        while i < len(gates):
            best = self.walk(gates, i)
            if best is None:
                i += 1
                continue
            _, k, new, tags = best
            if self.check and not same_function(gates[i:k + 1], new, self.rng):
                raise AssertionError("optimizer produced a non-equivalent rewrite")
            gates[i:k + 1] = new
            self.counts.update(tags)
            self.rewrites += 1
            changed = True
        return changed

    def rule5(self, n: int, gates: List[Gate]) -> bool:
        """Apply Rule 5 wherever its pattern appears literally."""
        changed, i = False, 0
        while i < len(gates):
            new = R.rule5_apply(Circuit(n, gates), i)
            if new is not None and self.score(new.gates) < self.score(gates):
                gates[:] = new.gates
                self.counts["Rule 5"] += 1
                self.rewrites += 1
                changed = True
            else:
                i += 1
        return changed


def optimize(circuit: Circuit, objective: str = "cost", window: int = 50,
             extra_gates: int = 1, cost: CostFn = quantum_cost,
             check: bool = True, max_passes: int = 100) -> OptimizationResult:
    """Reduce a reversible circuit with the transformation rules.

    objective    "cost" minimises quantum cost, then gate count;
                 "gates" minimises gate count, then quantum cost.
    window       how many gates a moving gate may travel past.
    extra_gates  how many extra gates (rule D2) one move may leave behind.
    check        verify every rewrite by simulation on the lines it touches.
    """
    search = _Search(objective, window, extra_gates, cost, check)
    gates = list(circuit.gates)
    passes = 0
    while passes < max_passes:
        passes += 1
        changed = search.sweep(gates)
        # The same search on the reversed circuit moves gates to the left.
        # Each gate is its own inverse, so the reversed circuit is the
        # inverse circuit and every rule stays valid.
        gates.reverse()
        changed |= search.sweep(gates)
        gates.reverse()
        changed |= search.rule5(circuit.n, gates)
        if not changed:
            break
    return OptimizationResult(circuit, Circuit(circuit.n, gates), search.counts,
                              search.rewrites, passes, cost)

In [ ]:
%%writefile bitsim.py
"""Bit-parallel simulation of reversible circuits (Phase 4).

revcircuit.py simulates one input at a time, which is too slow for a
24-line adder (16.7 million inputs). Here every line holds one large
integer whose bit s is the value of that line for input state s, so a gate
is applied to all 2^n inputs at once with a few integer operations.

This makes an exhaustive equivalence check practical up to about 26 lines.
"""
from __future__ import annotations

from typing import List

from revcircuit import Circuit


def input_vectors(n: int) -> List[int]:
    """vectors[i] has bit s equal to bit i of s, for s = 0 .. 2^n - 1."""
    size = 1 << n
    vectors = []
    for i in range(n):
        half = 1 << i
        v = ((1 << half) - 1) << half            # 'half' zeros, then 'half' ones
        length = 2 * half
        while length < size:                     # repeat the pattern to full length
            v |= v << length
            length *= 2
        vectors.append(v)
    return vectors


def output_vectors(circuit: Circuit) -> List[int]:
    """Truth vector of every output line over all 2^n inputs."""
    n = circuit.n
    mask = (1 << (1 << n)) - 1
    lines = input_vectors(n)
    for g in circuit:
        active = mask
        for l in g.pos:
            active &= lines[l]
        for l in g.neg:
            active &= ~lines[l]
        lines[g.target] ^= active & mask
    return lines


def equivalent_all_inputs(a: Circuit, b: Circuit) -> bool:
    """True when the two circuits agree on every line for every one of the 2^n inputs."""
    return a.n == b.n and output_vectors(a) == output_vectors(b)

In [ ]:
%%writefile adders.py
"""Reversible adders built from Feynman, Peres, TSG and R2 gates (Phase 4).

Two 4-bit adders are provided, each as a circuit of Toffoli-type gates:

  cla_adder()   carry look-ahead adder in the style of Kumar et al.,
                "An Optimised Design of a 4-bit Carry Look-Ahead Adder Using
                Novel Reversible Logic Gates", IACIS 2024.
  rca_adder()   ripple-carry adder from two Peres gates per bit.

IMPORTANT: the CLA netlist is a RECONSTRUCTION. The paper gives a block
diagram (its Fig. 6), not a netlist, and the diagram lets one signal feed
several blocks, which a reversible circuit cannot do without copy gates.
The circuit here follows the paper's building blocks and equations:

    PGA block   Peres(A_i, B_i, 0) gives P_i = A_i xor B_i and G_i = A_i B_i;
                Feynman(P_i, C_i) gives S_i = P_i xor C_i
    C1 = G0 xor P0 C0                               Peres + Feynman
    C2 = G1 xor P1 G0 xor P1 P0 C0                  Peres gates + TSG
    C3 = G2 xor P2 G1 xor P2 P1 G0 xor P2 P1 P0 C0  Peres gates + R2

and adds a Feynman copy gate wherever a signal is needed twice.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, TOF

# Quantum cost of each block as stated in the adder paper.
PAPER_BLOCK_COST = {"Feynman": 1, "Peres": 4, "TSG": 17, "R2": 5}

# Figures the adder paper reports for its own design (its Table I).
PAPER_REPORTED = {"gates": 20, "ancilla": 14, "garbage": 14, "cost": 70, "trlic": 118}


# ---------------------------------------------------------------------------
# Gate library: each block as Toffoli-type gates, and its defining function
# ---------------------------------------------------------------------------
def feynman(a: int, b: int) -> List[Gate]:
    """Feynman gate: (A, B) -> (A, A xor B)."""
    return [CNOT(a, b)]


def peres(a: int, b: int, c: int) -> List[Gate]:
    """Peres gate: (A, B, C) -> (A, A xor B, AB xor C)."""
    return [TOF(a, b, c), CNOT(a, b)]


def tsg(a: int, b: int, c: int, d: int) -> List[Gate]:
    """TSG gate of Thapliyal and Srinivas: (A, B, C, D) -> (A, Q, Q xor D, QD xor AB xor C)
    with Q = A'C' xor B'. The outputs appear on lines a, b, d, c in that order."""
    return [TOF(a, b, c), CNOT(a, b), G([c], [a], b), TOF(b, d, c), CNOT(b, d)]


def r2(a: int, b: int, c: int, d: int) -> List[Gate]:
    """R2 gate: (A, B, C, D) -> (A, B, C, D xor A xor B xor C)."""
    return [CNOT(a, d), CNOT(b, d), CNOT(c, d)]


def feynman_ref(a, b):
    return a, a ^ b


def peres_ref(a, b, c):
    return a, a ^ b, (a & b) ^ c


def tsg_ref(a, b, c, d):
    q = ((1 - a) & (1 - c)) ^ (1 - b)
    return a, q, q ^ d, (q & d) ^ (a & b) ^ c


def r2_ref(a, b, c, d):
    return a, b, c, d ^ a ^ b ^ c


# ---------------------------------------------------------------------------
# A design: circuit plus the meaning of its lines
# ---------------------------------------------------------------------------
@dataclass
class Design:
    name: str
    circuit: Circuit
    line_names: List[str]
    inputs: Dict[str, int]                 # primary input name -> line
    ancilla: List[int]                     # lines that must start at 0
    outputs: Dict[str, int]                # primary output name -> line
    blocks: List[Tuple[str, Tuple[int, ...]]] = field(default_factory=list)

    @property
    def n(self) -> int:
        return self.circuit.n

    @property
    def garbage(self) -> List[int]:
        """Every output line that is not a primary output."""
        used = set(self.outputs.values())
        return [l for l in range(self.n) if l not in used]

    def block_count(self) -> int:
        return len(self.blocks)

    def block_cost(self) -> int:
        """Quantum cost with the block costs stated in the adder paper."""
        return sum(PAPER_BLOCK_COST[kind] for kind, _ in self.blocks)

    def block_summary(self) -> str:
        kinds = [kind for kind, _ in self.blocks]
        return ", ".join(f"{kinds.count(k)} {k}" for k in ("Peres", "Feynman", "TSG", "R2")
                         if k in kinds)

    def with_circuit(self, circuit: Circuit, name: str) -> "Design":
        """The same interface around another circuit (e.g. an optimized one)."""
        return Design(name, circuit, self.line_names, self.inputs, self.ancilla,
                      self.outputs, [])

    def run(self, values: Dict[str, int]) -> Dict[str, int]:
        """Simulate with the given primary input bits; ancilla lines start at 0."""
        state = sum(values[name] << line for name, line in self.inputs.items())
        out = self.circuit.apply(state)
        return {name: (out >> line) & 1 for name, line in self.outputs.items()}


class _Builder:
    def __init__(self, names: Sequence[str]) -> None:
        self.names = list(names)
        self.index = {name: i for i, name in enumerate(self.names)}
        self.gates: List[Gate] = []
        self.blocks: List[Tuple[str, Tuple[int, ...]]] = []

    def add(self, kind: str, *line_names: str) -> None:
        lines = tuple(self.index[name] for name in line_names)
        maker = {"Feynman": feynman, "Peres": peres, "TSG": tsg, "R2": r2}[kind]
        self.gates += maker(*lines)
        self.blocks.append((kind, lines))


# ---------------------------------------------------------------------------
# Carry look-ahead adder (reconstruction of Kumar et al.)
# ---------------------------------------------------------------------------
def cla_adder(carry_out: bool = False) -> Design:
    """4-bit reversible carry look-ahead adder.

    carry_out=False  the paper's block inventory: sum bits S0..S3 only.
    carry_out=True   the last Feynman gate is replaced by a Peres gate that
                     also produces the final carry C4 = G3 xor P3 C3.
    """
    names = ([f"a{i}" for i in range(4)] + [f"b{i}" for i in range(4)] + ["c0"]
             + [f"g{i}" for i in range(4)]            # G_i
             + ["k0", "k1", "k2", "k3"]               # copies
             + [f"t{i}" for i in range(1, 7)]         # product terms
             + ["z"])                                 # constant input of the TSG gate
    b = _Builder(names)
    for i in range(4):                                # PGA, Peres part: b_i = P_i, g_i = G_i
        b.add("Peres", f"a{i}", f"b{i}", f"g{i}")
    b.add("Feynman", "c0", "k0")                      # copy C0
    b.add("Feynman", "b0", "c0")                      # PGA 0, Feynman part: c0 = S0
    b.add("Peres", "b0", "k0", "t1")                  # t1 = P0 C0
    b.add("Feynman", "g0", "k1")                      # copy G0
    b.add("Feynman", "t1", "k1")                      # k1 = C1 = G0 xor P0 C0
    b.add("Peres", "b1", "g0", "t2")                  # t2 = P1 G0
    b.add("Peres", "b1", "t1", "t3")                  # t3 = P1 P0 C0
    b.add("Feynman", "b1", "k1")                      # PGA 1, Feynman part: k1 = S1
    b.add("Feynman", "g1", "k2")                      # copy G1
    b.add("Feynman", "t2", "k3")                      # copy P1 G0
    b.add("TSG", "t3", "k3", "z", "k2")               # k2 = C2 (three-input XOR)
    b.add("Peres", "b2", "g1", "t4")                  # t4 = P2 G1
    b.add("Peres", "b2", "t2", "t5")                  # t5 = P2 P1 G0
    b.add("Peres", "b2", "t3", "t6")                  # t6 = P2 P1 P0 C0
    b.add("Feynman", "b2", "k2")                      # PGA 2, Feynman part: k2 = S2
    b.add("R2", "t4", "t5", "t6", "g2")               # g2 = C3 (four-input XOR)
    outputs = {"s0": "c0", "s1": "k1", "s2": "k2", "s3": "g2"}
    if carry_out:
        b.add("Peres", "b3", "g2", "g3")              # g2 = S3, g3 = C4
        outputs["c4"] = "g3"
    else:
        b.add("Feynman", "b3", "g2")                  # PGA 3, Feynman part: g2 = S3
    ix = b.index
    return Design(
        name="CLA with carry-out" if carry_out else "CLA (paper style)",
        circuit=Circuit(len(names), b.gates), line_names=names,
        inputs={n: ix[n] for n in names[:9]},
        ancilla=[ix[n] for n in names[9:]],
        outputs={k: ix[v] for k, v in outputs.items()}, blocks=b.blocks)


# ---------------------------------------------------------------------------
# Ripple-carry adder: two Peres gates per bit
# ---------------------------------------------------------------------------
def rca_adder() -> Design:
    """4-bit reversible ripple-carry adder with carry-out.

    Per bit: Peres(A_i, B_i, 0) gives P_i and G_i, then Peres(P_i, C_i, G_i)
    gives S_i and C_(i+1) = G_i xor P_i C_i.
    """
    names = ([f"a{i}" for i in range(4)] + [f"b{i}" for i in range(4)] + ["c0"]
             + [f"z{i}" for i in range(4)])
    b = _Builder(names)
    carry = "c0"
    outputs = {}
    for i in range(4):
        b.add("Peres", f"a{i}", f"b{i}", f"z{i}")
        b.add("Peres", f"b{i}", carry, f"z{i}")
        outputs[f"s{i}"] = carry
        carry = f"z{i}"
    outputs["c4"] = carry
    ix = b.index
    return Design(
        name="Ripple-carry (Peres)", circuit=Circuit(len(names), b.gates), line_names=names,
        inputs={n: ix[n] for n in names[:9]}, ancilla=[ix[n] for n in names[9:]],
        outputs={k: ix[v] for k, v in outputs.items()}, blocks=b.blocks)


# ---------------------------------------------------------------------------
# Checks and measures
# ---------------------------------------------------------------------------
def check_adder(design: Design) -> Tuple[int, int]:
    """Test all 512 input combinations. Returns (cases tested, cases wrong)."""
    wrong = 0
    for a in range(16):
        for b in range(16):
            for c0 in range(2):
                values = {f"a{i}": (a >> i) & 1 for i in range(4)}
                values.update({f"b{i}": (b >> i) & 1 for i in range(4)})
                values["c0"] = c0
                out = design.run(values)
                total = a + b + c0
                good = all(out[f"s{i}"] == (total >> i) & 1 for i in range(4))
                if "c4" in out:
                    good = good and out["c4"] == (total >> 4) & 1
                wrong += not good
    return 512, wrong


def add(design: Design, a: int, b: int, c0: int = 0):
    """Add two 4-bit numbers with a design. Returns (sum, carry); carry is None
    when the design has no carry output."""
    values = {f"a{i}": (a >> i) & 1 for i in range(4)}
    values.update({f"b{i}": (b >> i) & 1 for i in range(4)})
    values["c0"] = c0
    out = design.run(values)
    total = sum(out[f"s{i}"] << i for i in range(4))
    return total, out.get("c4")


def depth(circuit: Circuit) -> int:
    """Number of time steps when gates on different lines run in parallel."""
    level = [0] * circuit.n
    for g in circuit:
        step = 1 + max(level[l] for l in g.lines)
        for l in g.lines:
            level[l] = step
    return max(level, default=0)

In [ ]:
%%writefile bdd.py
"""A small binary decision diagram (BDD) package (Phase 8).

simplify.py needs to compare Boolean functions of the primary inputs
exactly. For a 4-bit adder it lists all 512 input combinations, but a 32-bit
adder has 2^65 of them. A reduced ordered BDD represents a function as a
graph in which equal functions share one node, so two functions are equal
exactly when their node numbers are equal. For adders the graphs stay small
when the variable order interleaves the operand bits: c0, a0, b0, a1, b1, ...

Only what the project needs is implemented: AND, XOR, NOT, constants and
variables. `Fn` wraps a node so that the operators &, ^, ~ and == work, which
lets the same simplification code run on integers or on BDDs.
"""
from __future__ import annotations

import sys
from typing import Dict, List, Tuple

sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))


class BDD:
    """Reduced ordered BDD over variables 0 .. nvars-1 (0 is the top variable)."""

    def __init__(self, nvars: int) -> None:
        self.nvars = nvars
        self.level: List[int] = [nvars, nvars]       # nodes 0 (false) and 1 (true)
        self.low: List[int] = [0, 1]
        self.high: List[int] = [0, 1]
        self._unique: Dict[Tuple[int, int, int], int] = {}
        self._and: Dict[Tuple[int, int], int] = {}
        self._xor: Dict[Tuple[int, int], int] = {}

    def _node(self, level: int, low: int, high: int) -> int:
        if low == high:
            return low
        key = (level, low, high)
        node = self._unique.get(key)
        if node is None:
            node = len(self.level)
            self.level.append(level)
            self.low.append(low)
            self.high.append(high)
            self._unique[key] = node
        return node

    def var(self, i: int) -> int:
        return self._node(i, 0, 1)

    def _split(self, u: int, level: int) -> Tuple[int, int]:
        if self.level[u] == level:
            return self.low[u], self.high[u]
        return u, u

    def AND(self, u: int, v: int) -> int:
        if u == 0 or v == 0:
            return 0
        if u == 1:
            return v
        if v == 1 or u == v:
            return u
        key = (u, v) if u < v else (v, u)
        r = self._and.get(key)
        if r is None:
            level = min(self.level[u], self.level[v])
            u0, u1 = self._split(u, level)
            v0, v1 = self._split(v, level)
            r = self._node(level, self.AND(u0, v0), self.AND(u1, v1))
            self._and[key] = r
        return r

    def XOR(self, u: int, v: int) -> int:
        if u == v:
            return 0
        if u == 0:
            return v
        if v == 0:
            return u
        key = (u, v) if u < v else (v, u)
        r = self._xor.get(key)
        if r is None:
            level = min(self.level[u], self.level[v])
            u0, u1 = self._split(u, level)
            v0, v1 = self._split(v, level)
            r = self._node(level, self.XOR(u0, v0), self.XOR(u1, v1))
            self._xor[key] = r
        return r

    def NOT(self, u: int) -> int:
        return self.XOR(u, 1)

    def evaluate(self, u: int, assignment) -> int:
        """Value of the function for a list of 0/1 values, one per variable."""
        while u > 1:
            u = self.high[u] if assignment[self.level[u]] else self.low[u]
        return u

    def size(self) -> int:
        return len(self.level)


class Fn:
    """A Boolean function held as a BDD node, with operator syntax."""

    __slots__ = ("bdd", "node")

    def __init__(self, bdd: BDD, node: int) -> None:
        self.bdd, self.node = bdd, node

    def __and__(self, other: "Fn") -> "Fn":
        return Fn(self.bdd, self.bdd.AND(self.node, other.node))

    def __xor__(self, other: "Fn") -> "Fn":
        return Fn(self.bdd, self.bdd.XOR(self.node, other.node))

    def __invert__(self) -> "Fn":
        return Fn(self.bdd, self.bdd.NOT(self.node))

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Fn) and self.node == other.node

    def __ne__(self, other: object) -> bool:
        return not self.__eq__(other)

    def __hash__(self) -> int:
        return hash(self.node)

    def __repr__(self) -> str:
        return f"Fn({self.node})"

In [ ]:
%%writefile simplify.py
"""Ancilla- and garbage-aware simplification (Phase 5).

The transformation rules of Feng and Li keep the function of a circuit the
same on EVERY line and for EVERY input. A practical circuit needs less:

  * ancilla lines always start at 0, and
  * garbage outputs are never read.

Two circuits are called design-equivalent here when they give the same
primary outputs for every value of the primary inputs, with all ancilla
lines starting at 0. The steps below keep design-equivalence. They are NOT
among the rules of the paper, whose authors leave the case with ancillary
bits open; they are heuristics of this project.

  constants  A gate that can never fire is removed, and a control that is
             always satisfied is dropped. "Never" and "always" refer to the
             valid inputs only (ancilla lines at 0).
  garbage    A gate is removed when its target is not a primary output and
             is not read later by any gate that matters.
  alias      A control is moved to another line that holds the same value at
             that point (for example the source of a copy), after commuting
             the gate earlier if needed. Copy gates then become garbage.
  xor        A CNOT whose control equals the XOR of two or three other lines
             is replaced by CNOTs from those lines, when that makes the gates
             that built the control garbage.
  rules      The Phase 3 optimizer (the paper's rules) runs in between.

The valid inputs are simulated all at once: each line holds one integer with
one bit per combination of the primary inputs. Every step is accepted only
if it lowers (cost, gate count, lines used), and with check=True the primary
outputs are compared with the original on all valid inputs after each step.
"""
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass, field
from itertools import combinations
from typing import Callable, Iterable, List, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate
from optimizer import circuit_cost, commute_tag, optimize, quantum_cost
from bitsim import input_vectors
from bdd import BDD, Fn

ALL_STEPS = ("constants", "garbage", "alias", "xor", "rules")


@dataclass
class SimplifyResult:
    original: Circuit
    circuit: Circuit
    inputs: List[int]
    ancilla: List[int]
    outputs: List[int]
    step_counts: Counter = field(default_factory=Counter)
    cost_fn: Callable[[Gate], int] = quantum_cost

    def _used(self, circuit: Circuit) -> set:
        touched = set()
        for g in circuit:
            touched |= g.lines
        return touched

    def ancilla_used(self, circuit: Circuit = None) -> int:
        """Ancilla lines that some gate still touches."""
        circuit = self.circuit if circuit is None else circuit
        return len(self._used(circuit) & set(self.ancilla))

    def lines_used(self, circuit: Circuit = None) -> int:
        circuit = self.circuit if circuit is None else circuit
        return len(self.inputs) + self.ancilla_used(circuit)

    def garbage(self, circuit: Circuit = None) -> int:
        """Used lines that are not primary outputs."""
        return self.lines_used(circuit) - len(self.outputs)

    def summary(self) -> str:
        def change(before, after):
            pct = 100.0 * (before - after) / before if before else 0.0
            return f"{before} -> {after} ({pct:.1f}% lower)"

        o, c = self.original, self.circuit
        lines = [
            f"Gate count:     {change(len(o), len(c))}",
            f"Quantum cost:   {change(circuit_cost(o, self.cost_fn), circuit_cost(c, self.cost_fn))}",
            f"Ancilla inputs: {change(self.ancilla_used(o), self.ancilla_used())}",
            f"Garbage lines:  {change(self.garbage(o), self.garbage())}",
        ]
        if self.step_counts:
            lines.append("Steps applied: " + ", ".join(
                f"{name} x{count}" for name, count in sorted(self.step_counts.items())))
        return "\n".join(lines)


class _Care:
    """Simulation over the valid inputs, and the simplification steps.

    Two interchangeable ways of holding "the value of a line for every valid
    input" are supported:

      bits   one integer with one bit per combination of the primary inputs.
             Exact, and fast up to about 17 primary inputs.
      bdd    one binary decision diagram per line. Exact for any number of
             inputs; practical when the diagrams stay small, as for adders.

    With the bdd backend a second, random simulation runs alongside. It is
    used only to find candidates quickly; every decision is taken on the
    exact values.
    """

    def __init__(self, n: int, inputs: Sequence[int], ancilla: Sequence[int],
                 outputs: Sequence[int], cost: Callable[[Gate], int],
                 backend: str = "auto", ancilla_ones: Sequence[int] = ()) -> None:
        if sorted(list(inputs) + list(ancilla)) != list(range(n)):
            raise ValueError("every line must be either a primary input or an ancilla")
        if not set(ancilla_ones) <= set(ancilla):
            raise ValueError("ancilla_ones must be a subset of ancilla")
        if backend == "auto":
            backend = "bits" if len(inputs) <= 17 else "bdd"
        if backend not in ("bits", "bdd"):
            raise ValueError('backend must be "bits", "bdd" or "auto"')
        self.n, self.outputs, self.cost, self.backend = n, list(outputs), cost, backend
        self.ancilla = set(ancilla)
        self.counts: Counter = Counter()
        k = len(inputs)
        if backend == "bits":
            self.zero, self.mask = 0, (1 << (1 << k)) - 1
            self.start = [0] * n
            for vector, line in zip(input_vectors(k), inputs):
                self.start[line] = vector
            self.sig_mask, self.sig_start = self.mask, self.start
        else:
            manager = BDD(k)
            self.zero, self.mask = Fn(manager, 0), Fn(manager, 1)
            self.start = [self.zero] * n
            for i, line in enumerate(inputs):          # variable order = order of `inputs`
                self.start[line] = Fn(manager, manager.var(i))
            rng = random.Random(12345)
            width = 1024
            self.sig_mask = (1 << width) - 1
            self.sig_start = [0] * n
            for line in inputs:
                self.sig_start[line] = rng.getrandbits(width)
        for line in ancilla_ones:
            self.start = list(self.start)
            self.sig_start = list(self.sig_start) if self.sig_start is not self.start else self.start
            self.start[line] = self.mask
            self.sig_start[line] = self.sig_mask

    # -- simulation ---------------------------------------------------------
    def activation(self, g: Gate, v: List, ones=None):
        """Where the gate fires: exact values, or signatures when `ones` is given."""
        act = self.mask if ones is None else ones
        for l in g.pos:
            act = act & v[l]
        for l in g.neg:
            act = act & ~v[l]
        return act if ones is None else act & ones

    def states(self, gates: Sequence[Gate]) -> List[List]:
        """Exact line values before each gate, and after the last one."""
        v = list(self.start)
        out = [list(v)]
        for g in gates:
            v[g.target] = v[g.target] ^ self.activation(g, v)
            out.append(list(v))
        return out

    def sig_states(self, gates: Sequence[Gate], exact: List[List]) -> List[List[int]]:
        """Integer signatures of the line values, used to look candidates up."""
        if self.backend == "bits":
            return exact
        v = list(self.sig_start)
        out = [list(v)]
        for g in gates:
            v[g.target] ^= self.activation(g, v, self.sig_mask)
            out.append(list(v))
        return out

    def signature(self, gates: Sequence[Gate]) -> Tuple:
        """The primary outputs for all valid inputs (exact)."""
        v = list(self.start)
        for g in gates:
            v[g.target] = v[g.target] ^ self.activation(g, v)
        return tuple(v[l] for l in self.outputs)

    def score(self, gates: Sequence[Gate]) -> Tuple[int, int, int]:
        touched = set()
        for g in gates:
            touched |= g.lines
        return (sum(self.cost(g) for g in gates), len(gates), len(touched & self.ancilla))

    # -- steps --------------------------------------------------------------
    def constants(self, gates: List[Gate]) -> List[Gate]:
        v = list(self.start)
        out = []
        for g in gates:
            act = self.activation(g, v)
            if act == self.zero:
                self.counts["constants: gate never fires"] += 1
                continue
            reduced = True
            while reduced:
                reduced = False
                for l in sorted(g.controls):
                    smaller = G(g.pos - {l}, g.neg - {l}, g.target)
                    if self.cost(smaller) <= self.cost(g) and self.activation(smaller, v) == act:
                        g, reduced = smaller, True
                        self.counts["constants: control dropped"] += 1
                        break
            v[g.target] = v[g.target] ^ act
            out.append(g)
        return out

    def garbage(self, gates: List[Gate], count: bool = True) -> List[Gate]:
        live = set(self.outputs)
        kept = []
        for g in reversed(gates):
            if g.target in live:
                kept.append(g)
                live |= g.controls
            elif count:
                self.counts["garbage: gate removed"] += 1
        return kept[::-1]

    def alias(self, gates: List[Gate]) -> Tuple[List[Gate], bool]:
        states = self.states(gates)
        sigs = self.sig_states(gates, states)
        base = self.score(gates)
        index_at = {}                                   # position -> {signature: [lines]}
        for j, g in enumerate(gates):
            i = j
            while True:
                v, sig = states[i], sigs[i]
                for u in sorted(g.controls):
                    if v[u] == self.zero or v[u] == self.mask:
                        continue
                    if i not in index_at:
                        table = {}
                        for line in range(self.n):
                            table.setdefault(sig[line], []).append(line)
                        index_at[i] = table
                    table = index_at[i]
                    complement = ~sig[u] & self.sig_mask if self.backend == "bdd" else ~v[u] & self.mask
                    for w in sorted(set(table.get(sig[u], [])) | set(table.get(complement, []))):
                        if w in g.lines:
                            continue
                        same = v[w] == v[u]
                        opposite = v[w] == (~v[u] & self.mask)
                        if not (same or opposite):
                            continue
                        positive = (u in g.pos) == same
                        pos = (g.pos - {u}) | ({w} if positive else set())
                        neg = (g.neg - {u}) | (set() if positive else {w})
                        trial = gates[:i] + [G(pos, neg, g.target)] + gates[i:j] + gates[j + 1:]
                        trial = self.garbage(trial, count=False)
                        if self.score(trial) < base:
                            self.counts["alias: control moved to an equal line"] += 1
                            return trial, True
                if i == 0 or commute_tag(g, gates[i - 1]) is None:
                    break
                i -= 1
        return gates, False

    def xor(self, gates: List[Gate]) -> Tuple[List[Gate], bool]:
        states = self.states(gates)
        sigs = self.sig_states(gates, states)
        base = self.score(gates)
        for j, g in enumerate(gates):
            if len(g.pos) != 1 or g.neg:
                continue
            (u,) = g.pos
            v, sig = states[j], sigs[j]
            lines = [l for l in range(self.n) if l not in (u, g.target) and v[l] != self.zero]
            table = {}
            for l in lines:
                table.setdefault(sig[l], []).append(l)

            def candidates():
                for x in lines:                          # pairs x < y
                    for y in table.get(sig[u] ^ sig[x], []):
                        if y > x:
                            yield (x, y)
                for a, x in enumerate(lines):            # triples x < y < z
                    for y in lines[a + 1:]:
                        for z in table.get(sig[u] ^ sig[x] ^ sig[y], []):
                            if z > y:
                                yield (x, y, z)

            for combo in candidates():
                total = self.zero
                for l in combo:
                    total = total ^ v[l]
                if total != v[u]:
                    continue
                trial = gates[:j] + [CNOT(l, g.target) for l in combo] + gates[j + 1:]
                trial = self.garbage(trial, count=False)
                if self.score(trial) < base:
                    self.counts["xor: control rebuilt from other lines"] += 1
                    return trial, True
        return gates, False

    def rules(self, gates: List[Gate]) -> List[Gate]:
        result = optimize(Circuit(self.n, gates), cost=self.cost)
        if self.score(result.circuit.gates) < self.score(gates):
            self.counts["rules: Phase 3 rewrites"] += result.rewrites
            return list(result.circuit.gates)
        return gates


def simplify(circuit: Circuit, inputs: Sequence[int], ancilla: Sequence[int],
             outputs: Sequence[int], steps: Iterable[str] = ALL_STEPS,
             cost: Callable[[Gate], int] = quantum_cost, check: bool = True,
             backend: str = "auto", ancilla_ones: Sequence[int] = (),
             rules_first: bool = True) -> SimplifyResult:
    """Simplify a circuit, using that ancilla lines start at a constant and
    that only the lines in `outputs` are read.

    inputs        lines that carry primary inputs. With the bdd backend their
                  order is the variable order; interleave operand bits.
    ancilla       lines that start at a constant (0 unless listed in ancilla_ones)
    outputs       lines that carry primary outputs; all other outputs are garbage
    steps         which steps to use, any of ALL_STEPS
    backend       "bits", "bdd", or "auto" (bits up to 17 primary inputs)
    ancilla_ones  ancilla lines that start at 1
    rules_first   run the rule-based optimizer before the other steps in each
                  round (default) or after them
    """
    steps = set(steps)
    if not steps <= set(ALL_STEPS):
        raise ValueError(f"steps must come from {ALL_STEPS}")
    care = _Care(circuit.n, inputs, ancilla, outputs, cost, backend, ancilla_ones)
    gates = list(circuit.gates)
    reference = care.signature(gates)

    def confirm(new: List[Gate]) -> List[Gate]:
        if check and care.signature(new) != reference:
            raise AssertionError("a simplification step changed a primary output")
        return new

    while True:
        before = care.score(gates)
        if "rules" in steps and rules_first:
            gates = confirm(care.rules(gates))
        if "constants" in steps:
            gates = confirm(care.constants(gates))
        if "garbage" in steps:
            gates = confirm(care.garbage(gates))
        for name, step in (("alias", care.alias), ("xor", care.xor)):
            changed = name in steps
            while changed:
                gates, changed = step(gates)
                confirm(gates)
        if "rules" in steps and not rules_first:
            gates = confirm(care.rules(gates))
        if care.score(gates) >= before:
            break
    return SimplifyResult(circuit, Circuit(circuit.n, gates), list(inputs), list(ancilla),
                          list(outputs), care.counts, cost)


def simplify_design(design, steps: Iterable[str] = ALL_STEPS, **kw) -> SimplifyResult:
    """simplify() for a Design from adders.py."""
    return simplify(design.circuit, list(design.inputs.values()), design.ancilla,
                    list(design.outputs.values()), steps, **kw)


# ---------------------------------------------------------------------------
# Counting a circuit the way the adder paper does
# ---------------------------------------------------------------------------
def block_view(circuit: Circuit) -> Tuple[int, int, Counter]:
    """Count blocks and price them as in the adder paper.

    A Toffoli gate next to a CNOT between its two controls is one Peres block
    (cost 4). Every other gate is one block at its own cost: Feynman or NOT 1,
    Toffoli-type gates by the project cost model.
    Returns (number of blocks, cost, how many blocks of each kind).
    """
    def peres_pair(a: Gate, b: Gate) -> bool:
        tof, cnot = (a, b) if len(a.controls) == 2 else (b, a)
        return (len(tof.pos) == 2 and not tof.neg and len(cnot.pos) == 1 and not cnot.neg
                and cnot.pos <= tof.pos and cnot.target in tof.pos)

    kinds: Counter = Counter()
    blocks = cost = i = 0
    gates = circuit.gates
    while i < len(gates):
        if i + 1 < len(gates) and peres_pair(gates[i], gates[i + 1]):
            kinds["Peres"] += 1
            cost += 4
            i += 2
        else:
            g = gates[i]
            c = len(g.controls)
            kinds["NOT" if c == 0 else "Feynman" if c == 1 and g.pos else "Toffoli-type"] += 1
            cost += quantum_cost(g)
            i += 1
        blocks += 1
    return blocks, cost, kinds

In [ ]:
%%writefile nadders.py
"""Reversible n-bit adders from the literature (Phase 8).

Six designs, each for any width n (cla4_blocks needs n divisible by 4):

  Garbage style: only the sum and carry lines are read at the end.
    cla4_blocks   4-bit carry look-ahead blocks in the style of Kumar et al.
                  (IACIS 2024), with the carry rippling from block to block.
                  The 4-bit block is a RECONSTRUCTION; see adders.py.
    tsg_rca       ripple-carry adder with one TSG gate per bit
                  (Thapliyal and Srinivas).
    peres_rca     ripple-carry adder with two Peres gates per bit.

  Clean style: every line is part of the specification. Operands are
  restored and every ancilla returns to 0, as quantum algorithms require.
    cuccaro       MAJ/UMA ripple-carry adder of Cuccaro et al., simple version.
    vbe           CARRY/SUM adder of Vedral, Barenco and Ekert.
    draper_qcla   out-of-place carry look-ahead adder of Draper et al.,
                  without carry input; logarithmic depth.

prove_adder() checks a design exactly for any width with decision diagrams.
"""
from __future__ import annotations

import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, TOF
from adders import feynman, peres, r2, tsg
from bdd import BDD, Fn


@dataclass
class AdderDesign:
    name: str
    style: str                              # "garbage" or "clean"
    bits: int
    circuit: Circuit
    line_names: List[str]
    inputs: Dict[str, int]                  # primary inputs, in decision-diagram order
    ancilla: List[int]                      # lines that start at 0
    sum_lines: List[int]                    # line of S0, S1, ...
    carry_line: Optional[int]               # line of the carry-out
    carry_in: Optional[int]                 # line of the carry input, if it is a primary input
    restored: Dict[int, str]                # clean style: line -> input name it must equal again
    blocks: List[Tuple[str, tuple]] = field(default_factory=list)

    @property
    def n(self) -> int:
        return self.circuit.n

    @property
    def required(self) -> List[int]:
        """Lines whose final value is part of the specification."""
        if self.style == "clean":
            return list(range(self.n))
        return self.sum_lines + ([self.carry_line] if self.carry_line is not None else [])

    def with_circuit(self, circuit: Circuit) -> "AdderDesign":
        return AdderDesign(self.name, self.style, self.bits, circuit, self.line_names, self.inputs,
                           self.ancilla, self.sum_lines, self.carry_line, self.carry_in,
                           self.restored, [])


class _Lines:
    def __init__(self) -> None:
        self.names: List[str] = []
        self.index: Dict[str, int] = {}

    def add(self, *names: str) -> None:
        for name in names:
            self.index[name] = len(self.names)
            self.names.append(name)

    def __getitem__(self, name: str) -> int:
        return self.index[name]


def _interleaved(lines: _Lines, n: int, carry: Optional[str]) -> Dict[str, int]:
    order = ([carry] if carry else []) + [x for i in range(n) for x in (f"a{i}", f"b{i}")]
    return {name: lines[name] for name in order}


# ---------------------------------------------------------------------------
# Garbage-style designs
# ---------------------------------------------------------------------------
def cla4_blocks(n: int) -> AdderDesign:
    """Kumar-style 4-bit look-ahead blocks; the carry ripples between blocks."""
    if n % 4:
        raise ValueError("n must be a multiple of 4")
    L = _Lines()
    L.add(*[f"a{i}" for i in range(n)], *[f"b{i}" for i in range(n)], "c0",
          *[f"g{i}" for i in range(n)])
    for j in range(n // 4):
        L.add(*[f"k{m}_{j}" for m in range(4)], *[f"t{m}_{j}" for m in range(1, 7)], f"z_{j}")
    gates: List[Gate] = []
    blocks = []

    def add(kind, *names):
        ids = tuple(L[x] for x in names)
        gates.extend({"Feynman": feynman, "Peres": peres, "TSG": tsg, "R2": r2}[kind](*ids))
        blocks.append((kind, ids))

    sums = []
    cin = "c0"
    for j in range(n // 4):
        a = [f"a{4 * j + m}" for m in range(4)]
        b = [f"b{4 * j + m}" for m in range(4)]
        g = [f"g{4 * j + m}" for m in range(4)]
        k = [f"k{m}_{j}" for m in range(4)]
        t = [None] + [f"t{m}_{j}" for m in range(1, 7)]
        for m in range(4):
            add("Peres", a[m], b[m], g[m])              # b = P, g = G
        add("Feynman", cin, k[0])                       # copy the carry input
        add("Feynman", b[0], cin)                       # cin line = S(4j)
        add("Peres", b[0], k[0], t[1])                  # t1 = P0 C
        add("Feynman", g[0], k[1])
        add("Feynman", t[1], k[1])                      # k1 = C1
        add("Peres", b[1], g[0], t[2])
        add("Peres", b[1], t[1], t[3])
        add("Feynman", b[1], k[1])                      # k1 = S(4j+1)
        add("Feynman", g[1], k[2])
        add("Feynman", t[2], k[3])
        add("TSG", t[3], k[3], f"z_{j}", k[2])          # k2 = C2
        add("Peres", b[2], g[1], t[4])
        add("Peres", b[2], t[2], t[5])
        add("Peres", b[2], t[3], t[6])
        add("Feynman", b[2], k[2])                      # k2 = S(4j+2)
        add("R2", t[4], t[5], t[6], g[2])               # g2 = C3
        add("Peres", b[3], g[2], g[3])                  # g2 = S(4j+3), g3 = C(4j+4)
        sums += [L[cin], L[k[1]], L[k[2]], L[g[2]]]
        cin = g[3]
    inputs = _interleaved(L, n, "c0")
    ancilla = [i for i, name in enumerate(L.names) if name not in inputs]
    return AdderDesign(f"CLA blocks (Kumar style), {n} bits", "garbage", n,
                       Circuit(len(L.names), gates), L.names, inputs, ancilla, sums, L[cin],
                       L["c0"], {}, blocks)


def tsg_rca(n: int) -> AdderDesign:
    """One TSG gate per bit: TSG(A, B, 0, C) gives the sum on the carry line
    and the next carry on the constant line."""
    L = _Lines()
    L.add(*[f"a{i}" for i in range(n)], *[f"b{i}" for i in range(n)], "c0",
          *[f"z{i}" for i in range(n)])
    gates, blocks, sums = [], [], []
    carry = "c0"
    for i in range(n):
        ids = (L[f"a{i}"], L[f"b{i}"], L[f"z{i}"], L[carry])
        gates += tsg(*ids)
        blocks.append(("TSG", ids))
        sums.append(L[carry])
        carry = f"z{i}"
    inputs = _interleaved(L, n, "c0")
    ancilla = [L[f"z{i}"] for i in range(n)]
    return AdderDesign(f"TSG ripple-carry, {n} bits", "garbage", n, Circuit(len(L.names), gates),
                       L.names, inputs, ancilla, sums, L[carry], L["c0"], {}, blocks)


def peres_rca(n: int) -> AdderDesign:
    """Two Peres gates per bit."""
    L = _Lines()
    L.add(*[f"a{i}" for i in range(n)], *[f"b{i}" for i in range(n)], "c0",
          *[f"z{i}" for i in range(n)])
    gates, blocks, sums = [], [], []
    carry = "c0"
    for i in range(n):
        for ids in ((L[f"a{i}"], L[f"b{i}"], L[f"z{i}"]), (L[f"b{i}"], L[carry], L[f"z{i}"])):
            gates += peres(*ids)
            blocks.append(("Peres", ids))
        sums.append(L[carry])
        carry = f"z{i}"
    inputs = _interleaved(L, n, "c0")
    ancilla = [L[f"z{i}"] for i in range(n)]
    return AdderDesign(f"Peres ripple-carry, {n} bits", "garbage", n, Circuit(len(L.names), gates),
                       L.names, inputs, ancilla, sums, L[carry], L["c0"], {}, blocks)


# ---------------------------------------------------------------------------
# Clean designs
# ---------------------------------------------------------------------------
def cuccaro(n: int) -> AdderDesign:
    """MAJ/UMA ripple-carry adder of Cuccaro et al. (simple version).

    MAJ(x, y, w): CNOT(w, y), CNOT(w, x), Toffoli(x, y -> w).
    UMA(x, y, w): Toffoli(x, y -> w), CNOT(w, x), CNOT(x, y).
    The sum replaces b, a is restored, the ancilla c0 returns to 0 and the
    carry-out is added to z.
    """
    L = _Lines()
    L.add("c0", *[x for i in range(n) for x in (f"b{i}", f"a{i}")], "z")
    gates: List[Gate] = []

    def chain(i):
        x = "c0" if i == 0 else f"a{i - 1}"
        return L[x], L[f"b{i}"], L[f"a{i}"]

    for i in range(n):
        x, y, w = chain(i)
        gates += [CNOT(w, y), CNOT(w, x), TOF(x, y, w)]
    gates.append(CNOT(L[f"a{n - 1}"], L["z"]))
    for i in reversed(range(n)):
        x, y, w = chain(i)
        gates += [TOF(x, y, w), CNOT(w, x), CNOT(x, y)]
    inputs = _interleaved(L, n, None)
    return AdderDesign(f"Cuccaro ripple-carry, {n} bits", "clean", n, Circuit(len(L.names), gates),
                       L.names, inputs, [L["c0"], L["z"]], [L[f"b{i}"] for i in range(n)], L["z"],
                       None, {L[f"a{i}"]: f"a{i}" for i in range(n)})


def vbe(n: int) -> AdderDesign:
    """CARRY/SUM adder of Vedral, Barenco and Ekert.

    CARRY(c, a, b, c'): Toffoli(a, b -> c'), CNOT(a, b), Toffoli(c, b -> c').
    SUM(c, a, b): CNOT(a, b), CNOT(c, b).
    The sum replaces b, with the carry-out on the extra line b_n; a is
    restored and all carry lines return to 0.
    """
    L = _Lines()
    L.add(*[x for i in range(n) for x in (f"c{i}", f"a{i}", f"b{i}")], f"b{n}")

    def carry(i):
        c, a, b = L[f"c{i}"], L[f"a{i}"], L[f"b{i}"]
        nxt = L[f"c{i + 1}"] if i + 1 < n else L[f"b{n}"]
        return [TOF(a, b, nxt), CNOT(a, b), TOF(c, b, nxt)]

    def total(i):
        return [CNOT(L[f"a{i}"], L[f"b{i}"]), CNOT(L[f"c{i}"], L[f"b{i}"])]

    gates: List[Gate] = []
    for i in range(n):
        gates += carry(i)
    gates.append(CNOT(L[f"a{n - 1}"], L[f"b{n - 1}"]))
    gates += total(n - 1)
    for i in reversed(range(n - 1)):
        gates += carry(i)[::-1]
        gates += total(i)
    inputs = _interleaved(L, n, None)
    ancilla = [L[f"c{i}"] for i in range(n)] + [L[f"b{n}"]]
    return AdderDesign(f"VBE ripple-carry, {n} bits", "clean", n, Circuit(len(L.names), gates),
                       L.names, inputs, ancilla, [L[f"b{i}"] for i in range(n)], L[f"b{n}"], None,
                       {L[f"a{i}"]: f"a{i}" for i in range(n)})


def draper_qcla(n: int) -> AdderDesign:
    """Out-of-place carry look-ahead adder of Draper, Kutin, Rains and Svore,
    without carry input.

    Z[0..n] receives the sum and the carry-out. A and B are restored and the
    ancilla lines P_t[m] return to 0. G[j] is kept on Z[j] and P_0[i] on B[i].
    """
    if n < 2:
        raise ValueError("n must be at least 2")
    log = n.bit_length() - 1                                  # floor(log2 n)
    L = _Lines()
    L.add(*[x for i in range(n) for x in (f"a{i}", f"b{i}")], *[f"z{i}" for i in range(n + 1)])
    for t in range(1, log):
        L.add(*[f"p{t}_{m}" for m in range(1, n >> t)])

    def P(t, m):
        return L[f"b{m}"] if t == 0 else L[f"p{t}_{m}"]

    def Gn(j):
        return L[f"z{j}"]

    gates: List[Gate] = []
    for i in range(n):
        gates.append(TOF(L[f"a{i}"], L[f"b{i}"], Gn(i + 1)))             # generate bits
    for i in range(1, n):
        gates.append(CNOT(L[f"a{i}"], L[f"b{i}"]))                       # propagate bits
    p_rounds = [TOF(P(t - 1, 2 * m), P(t - 1, 2 * m + 1), P(t, m))
                for t in range(1, log) for m in range(1, n >> t)]
    gates += p_rounds
    for t in range(1, log + 1):                                          # G-rounds
        for m in range(0, n >> t):
            gates.append(TOF(Gn((m << t) + (1 << (t - 1))), P(t - 1, 2 * m + 1), Gn((m << t) + (1 << t))))
    top = 0
    while 3 * (1 << (top + 1)) <= 2 * n:                                 # floor(log2(2n/3))
        top += 1
    for t in range(top, 0, -1):                                          # C-rounds
        for m in range(1, (n - (1 << (t - 1))) // (1 << t) + 1):
            gates.append(TOF(Gn(m << t), P(t - 1, 2 * m), Gn((m << t) + (1 << (t - 1)))))
    gates += p_rounds[::-1]                                              # undo the P-rounds
    for i in range(n):
        gates.append(CNOT(L[f"b{i}"], L[f"z{i}"]))
    gates.append(CNOT(L["a0"], L["z0"]))
    for i in range(1, n):
        gates.append(CNOT(L[f"a{i}"], L[f"b{i}"]))                       # restore B
    inputs = _interleaved(L, n, None)
    ancilla = [i for i, name in enumerate(L.names) if name not in inputs]
    restored = {L[f"a{i}"]: f"a{i}" for i in range(n)}
    restored.update({L[f"b{i}"]: f"b{i}" for i in range(n)})
    return AdderDesign(f"Draper carry look-ahead, {n} bits", "clean", n,
                       Circuit(len(L.names), gates), L.names, inputs, ancilla,
                       [L[f"z{i}"] for i in range(n)], L[f"z{n}"], None, restored)


DESIGNS = {"cla4_blocks": cla4_blocks, "tsg_rca": tsg_rca, "peres_rca": peres_rca,
           "cuccaro": cuccaro, "vbe": vbe, "draper_qcla": draper_qcla}


# ---------------------------------------------------------------------------
# Exact proof and independent random test
# ---------------------------------------------------------------------------
def prove_adder(design: AdderDesign) -> bool:
    """Exact check for every input: sum, carry, restored lines, clean ancillas.

    All lines are simulated as decision diagrams over the primary inputs and
    compared with a reference adder built from the textbook recurrence.
    """
    names = list(design.inputs)
    bdd = BDD(len(names))
    var = {name: Fn(bdd, bdd.var(i)) for i, name in enumerate(names)}
    zero = Fn(bdd, 0)
    v = [zero] * design.n
    for name, line in design.inputs.items():
        v[line] = var[name]
    for g in design.circuit:
        act = Fn(bdd, 1)
        for l in g.pos:
            act = act & v[l]
        for l in g.neg:
            act = act & ~v[l]
        v[g.target] = v[g.target] ^ act
    carry = var["c0"] if design.carry_in is not None else zero
    for i in range(design.bits):
        a, b = var[f"a{i}"], var[f"b{i}"]
        if v[design.sum_lines[i]] != a ^ b ^ carry:
            return False
        carry = (a & b) ^ ((a ^ b) & carry)
    if design.carry_line is not None and v[design.carry_line] != carry:
        return False
    if design.style == "clean":
        for line, name in design.restored.items():
            if v[line] != var[name]:
                return False
        special = set(design.sum_lines) | {design.carry_line} | set(design.restored)
        if any(v[l] != zero for l in design.ancilla if l not in special):
            return False
    return True


def random_test(design: AdderDesign, trials: int = 2000, seed: int = 1) -> int:
    """Independent check with the plain simulator. Returns the number of wrong sums."""
    rng = random.Random(seed)
    n, wrong = design.bits, 0
    full = (1 << n) - 1
    corner = [(0, 0, 0), (full, 1, 0), (full, full, 1), (full, 0, 1), (1, full, 0)]
    for k in range(trials):
        a, b, c = corner[k] if k < len(corner) else (rng.getrandbits(n), rng.getrandbits(n), rng.getrandbits(1))
        if design.carry_in is None:
            c = 0
        state = 0
        for i in range(n):
            state |= ((a >> i) & 1) << design.inputs[f"a{i}"]
            state |= ((b >> i) & 1) << design.inputs[f"b{i}"]
        if design.carry_in is not None:
            state |= c << design.carry_in
        out = design.circuit.apply(state)
        total = a + b + c
        got = sum(((out >> line) & 1) << i for i, line in enumerate(design.sum_lines))
        if design.carry_line is not None:
            got |= ((out >> design.carry_line) & 1) << n
        else:
            total &= full
        wrong += got != total
    return wrong


# ---------------------------------------------------------------------------
# Measures
# ---------------------------------------------------------------------------
def toffoli_count(circuit: Circuit) -> int:
    """Gates with exactly two controls. Larger gates are not expected in adders."""
    if any(len(g.controls) > 2 for g in circuit):
        raise ValueError("a gate has more than two controls; T-count is not defined here")
    return sum(1 for g in circuit if len(g.controls) == 2)


def t_count(circuit: Circuit) -> int:
    """7 T gates per two-control Toffoli-type gate; NOT and CNOT are free."""
    return 7 * toffoli_count(circuit)


def toffoli_depth(circuit: Circuit) -> int:
    """Depth counting only two-control gates; NOT and CNOT take no time."""
    level = [0] * circuit.n
    for g in circuit:
        step = max(level[l] for l in g.lines) + (1 if len(g.controls) >= 2 else 0)
        for l in g.lines:
            level[l] = step
    return max(level, default=0)


def used_ancilla(design: AdderDesign, circuit: Optional[Circuit] = None) -> int:
    circuit = design.circuit if circuit is None else circuit
    touched = set()
    for g in circuit:
        touched |= g.lines
    return len(touched & set(design.ancilla))

In [ ]:
%%writefile revlib.py
"""RevLib benchmark circuits (Phase 8).

Reads circuits in the RevLib ".real" format and prepares them for the
rule-based optimizer and for the ancilla- and garbage-aware simplifier.

The 20 files used here are RevLib circuits (www.revlib.org) as distributed
with the test suite of the MQT QCEC tool, release v1.10.0:
    https://github.com/cda-tum/mqt-qcec/tree/v1.10.0/test/circuits/original
fetch() downloads them with git when they are not present.

Constant inputs are taken from the ".constants" line of each file. The
".garbage" line is used for the four files that also name their outputs
(garbage outputs are called "g" there). In the other files, which come from
an ESOP-based synthesis, the garbage string is not ordered consistently, so
the function outputs f are taken as primary outputs and the input lines x as
garbage; this is the evident intent of those files.
"""
from __future__ import annotations

import os
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional

from revcircuit import CNOT, Circuit, G, Gate

SOURCE = "https://github.com/cda-tum/mqt-qcec"
TAG = "v1.10.0"
SUBDIR = "test/circuits/original"


@dataclass
class Benchmark:
    name: str
    circuit: Circuit
    line_names: List[str]
    inputs: List[int]                  # lines with a free primary input
    ancilla: List[int]                 # lines with a constant input
    ancilla_ones: List[int]            # constant inputs that are 1
    outputs: Optional[List[int]]       # primary outputs, or None when the file gives none
    family: str                        # "esop", "bdd" or "plain"


def fetch(folder: str = "revlib") -> str:
    """Make sure the RevLib files are in `folder`; download them if needed."""
    if os.path.isdir(folder) and any(f.endswith(".real") for f in os.listdir(folder)):
        return folder
    clone = folder + "_clone"
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", "--depth", "1",
                    "--branch", TAG, SOURCE, clone], check=True)
    os.makedirs(folder, exist_ok=True)
    source = os.path.join(clone, SUBDIR)
    for name in os.listdir(source):
        if name.endswith(".real"):
            with open(os.path.join(source, name), encoding="utf-8") as src, \
                    open(os.path.join(folder, name), "w", encoding="utf-8") as dst:
                dst.write(src.read())
    return folder


def parse_real(path: str) -> Optional[Benchmark]:
    """Read one .real file. Returns None if it uses gates other than Toffoli,
    Fredkin and Peres gates (for example controlled-V gates)."""
    header: Dict[str, List[str]] = {}
    body: List[List[str]] = []
    in_body = False
    with open(path, encoding="utf-8") as handle:
        for raw in handle:
            line = raw.split("#")[0].strip()
            if not line:
                continue
            if line.lower() == ".begin":
                in_body = True
            elif line.lower() == ".end":
                in_body = False
            elif in_body:
                body.append(line.split())
            elif line.startswith("."):
                parts = line.split()
                header[parts[0].lower()] = parts[1:]
    names = header[".variables"]
    n = len(names)
    index = {name: i for i, name in enumerate(names)}

    def control(token: str):
        return (index[token[1:]], False) if token.startswith("-") else (index[token], True)

    gates: List[Gate] = []
    for tokens in body:
        kind, args = tokens[0].lower(), tokens[1:]
        if kind.startswith("t"):                                   # multiple-control Toffoli
            controls = [control(x) for x in args[:-1]]
            gates.append(G([l for l, p in controls if p], [l for l, p in controls if not p],
                           index[args[-1]]))
        elif kind.startswith("p"):                                 # Peres: Toffoli, then CNOT
            a, b, c = (index[x] for x in args)
            gates += [G([a, b], [], c), CNOT(a, b)]
        elif kind.startswith("f"):                                 # Fredkin: controlled swap
            controls = [control(x) for x in args[:-2]]
            x, y = index[args[-2]], index[args[-1]]
            pos = [l for l, p in controls if p]
            neg = [l for l, p in controls if not p]
            gates += [CNOT(y, x), G(pos + [x], neg, y), CNOT(y, x)]
        else:
            return None
    constants = "".join(header.get(".constants", ["-" * n]))
    garbage = "".join(header.get(".garbage", ["-" * n]))
    inputs = [i for i in range(n) if constants[i] == "-"]
    ancilla = [i for i in range(n) if constants[i] != "-"]
    ones = [i for i in range(n) if constants[i] == "1"]
    out_names = header.get(".outputs", [])
    if ".constants" not in header:
        family, outputs = "plain", None
    elif "g" in out_names:
        family, outputs = "bdd", [i for i in range(n) if garbage[i] == "-"]
    else:
        family, outputs = "esop", list(ancilla)
    name = os.path.basename(path)[:-len(".real")]
    return Benchmark(name, Circuit(n, gates), names, inputs, ancilla, ones, outputs, family)


def load_all(folder: str = "revlib") -> List[Benchmark]:
    """All usable benchmarks in the folder, sorted by name."""
    out = []
    for name in sorted(os.listdir(folder)):
        if name.endswith(".real"):
            bench = parse_real(os.path.join(folder, name))
            if bench is not None:
                out.append(bench)
    return out


# ---------------------------------------------------------------------------
# RevLib quantum cost
# ---------------------------------------------------------------------------
def _table(size: int, free: int) -> int:
    """Cost of a Toffoli gate on `size` lines when `free` other lines exist.

    This is the table used by RevLib. It reproduces the quantum cost printed
    in the header of 13 of the 14 files that state one; the last differs by 24.
    """
    if size <= 2:
        return 1
    if size == 3:
        return 5
    if size == 4:
        return 13
    if size == 5:
        return 26 if free >= 2 else 29
    rich = {6: (3, 38), 7: (4, 50), 8: (5, 62), 9: (6, 74), 10: (7, 86)}
    some = {6: 52, 7: 80, 8: 100, 9: 128, 10: 152}
    if size <= 10:
        need, low = rich[size]
        return low if free >= need else some[size] if free >= 1 else (1 << size) - 3
    return 12 * size - 34 if free >= size - 3 else 24 * size - 88 if free >= 1 else (1 << size) - 3


def revlib_cost(n: int):
    """Cost function for circuits on n lines, RevLib convention. Negative
    controls are priced like positive ones, except that a gate whose controls
    are all negative costs one more (the convention of this project)."""
    def cost(gate: Gate) -> int:
        size = len(gate.pos) + len(gate.neg) + 1
        extra = 1 if (gate.neg and not gate.pos) else 0
        return _table(size, n - size) + extra
    return cost

In [ ]:
%%writefile verilog.py
"""Verilog for the reversible adders (Phase 6).

Four text files are produced:

  rev_gates.v       the reversible gate library: NOT, Feynman, Toffoli,
                    Peres, TSG and R2, each with as many outputs as inputs
  rev_adders.v      one module per adder, built only from library gates
  tb_rev_adders.v   a self-checking testbench for the adders: a few named
                    cases for the waveform, then all 512 input combinations
  tb_rev_gates.v    a self-checking testbench for the gate library

The adder modules are structural. Every gate instance takes the current
wire of each of its lines and drives a new wire, so the netlist reads in the
same order as the reversible circuit. Ancilla lines are tied to 0. Outputs
that are not sum or carry bits are garbage and are left unconnected.

A synthesis tool treats these modules as ordinary combinational logic. The
simulation confirms that the circuits add correctly; it says nothing about
the energy behaviour of reversible logic.
"""
from __future__ import annotations

import os
import re
import shutil
import subprocess
import tempfile
from collections import Counter
from typing import Dict, List, Sequence, Tuple

from revcircuit import Circuit, Gate
from adders import Design, cla_adder, rca_adder
from simplify import simplify_design

Block = Tuple[str, tuple]

LIBRARY = """\
// Reversible gate library.
// Every gate has as many outputs as inputs and realises a one-to-one mapping.
`timescale 1ns / 1ps

// NOT gate: P = A'
module rev_not (input a, output p);
  assign p = ~a;
endmodule

// Feynman (CNOT) gate: P = A, Q = A xor B
module feynman_gate (input a, input b, output p, output q);
  assign p = a;
  assign q = a ^ b;
endmodule

// Toffoli gate: P = A, Q = B, R = AB xor C
module toffoli_gate (input a, input b, input c, output p, output q, output r);
  assign p = a;
  assign q = b;
  assign r = (a & b) ^ c;
endmodule

// Peres gate: P = A, Q = A xor B, R = AB xor C
module peres_gate (input a, input b, input c, output p, output q, output r);
  assign p = a;
  assign q = a ^ b;
  assign r = (a & b) ^ c;
endmodule

// TSG gate: P = A, Q = A'C' xor B', R = Q xor D, S = QD xor AB xor C
module tsg_gate (input a, input b, input c, input d,
                 output p, output q, output r, output s);
  assign p = a;
  assign q = (~a & ~c) ^ ~b;
  assign r = q ^ d;
  assign s = (q & d) ^ (a & b) ^ c;
endmodule

// R2 gate: P = A, Q = B, R = C, S = D xor A xor B xor C
module r2_gate (input a, input b, input c, input d,
                output p, output q, output r, output s);
  assign p = a;
  assign q = b;
  assign r = c;
  assign s = d ^ a ^ b ^ c;
endmodule
"""

# kind -> (module name, input ports, output ports, which line each output drives)
_PORTS = {
    "NOT": ("rev_not", "a", "p", (0,)),
    "Feynman": ("feynman_gate", "ab", "pq", (0, 1)),
    "Toffoli": ("toffoli_gate", "abc", "pqr", (0, 1, 2)),
    "Peres": ("peres_gate", "abc", "pqr", (0, 1, 2)),
    "TSG": ("tsg_gate", "abcd", "pqrs", (0, 1, 3, 2)),      # R drives line d, S drives line c
    "R2": ("r2_gate", "abcd", "pqrs", (0, 1, 2, 3)),
}


def blocks_from_circuit(circuit: Circuit) -> List[Block]:
    """Group Toffoli-type gates into library gates.

    A Toffoli gate followed directly by a CNOT from one of its controls to
    the other is one Peres gate. A gate the library does not have (negative
    controls, or more than two controls) is kept as ("MCT", (gate,)).
    """
    def plain(g: Gate, controls: int) -> bool:
        return len(g.pos) == controls and not g.neg

    blocks: List[Block] = []
    gates, i = circuit.gates, 0
    while i < len(gates):
        g = gates[i]
        nxt = gates[i + 1] if i + 1 < len(gates) else None
        if (plain(g, 2) and nxt is not None and plain(nxt, 1)
                and nxt.pos <= g.pos and nxt.target in g.pos):
            (a,) = nxt.pos
            blocks.append(("Peres", (a, nxt.target, g.target)))
            i += 2
            continue
        if plain(g, 2):
            blocks.append(("Toffoli", (*sorted(g.pos), g.target)))
        elif plain(g, 1):
            blocks.append(("Feynman", (*g.pos, g.target)))
        elif plain(g, 0):
            blocks.append(("NOT", (g.target,)))
        else:
            blocks.append(("MCT", (g,)))
        i += 1
    return blocks


def adder_module(name: str, design: Design, blocks: Sequence[Block], note: str) -> str:
    """Verilog module for a 4-bit adder design given as a list of blocks."""
    names = design.line_names
    has_carry = "c4" in design.outputs
    port_of = {f"a{i}": f"a[{i}]" for i in range(4)}
    port_of.update({f"b{i}": f"b[{i}]" for i in range(4)})
    port_of["c0"] = "cin"

    used = set(design.inputs.values()) | set(design.outputs.values())
    for kind, lines in blocks:
        used |= lines[0].lines if kind == "MCT" else set(lines)

    version: Dict[int, int] = {l: 0 for l in used}
    wires: List[str] = []
    body: List[str] = []

    def current(line: int) -> str:
        return f"{names[line]}_{version[line]}"

    def advance(line: int) -> str:
        version[line] += 1
        wires.append(current(line))
        return current(line)

    for line in sorted(used):
        wires.append(current(line))
        source = port_of.get(names[line], "1'b0")
        kind = "primary input" if names[line] in port_of else "ancilla, constant 0"
        body.append(f"  assign {current(line)} = {source};".ljust(30) + f"// {kind}")
    body.append("")

    counts: Counter = Counter()
    for index, (kind, lines) in enumerate(blocks, 1):
        counts[kind] += 1
        if kind == "MCT":
            g = lines[0]
            terms = [current(l) for l in sorted(g.pos)] + ["~" + current(l) for l in sorted(g.neg)]
            old = current(g.target)
            body.append(f"  assign {advance(g.target)} = {old} ^ ({' & '.join(terms)});"
                        f"   // gate {index}: Toffoli-type gate with negative control")
            continue
        module, ins, outs, drives = _PORTS[kind]
        inputs = [f".{port}({current(l)})" for port, l in zip(ins, lines)]
        outputs = [f".{port}({advance(lines[k])})" for port, k in zip(outs, drives)]
        body.append(f"  {module:<13} u{index} ({', '.join(inputs + outputs)});")
    body.append("")

    for i in range(4):
        body.append(f"  assign sum[{i}] = {current(design.outputs[f's{i}'])};")
    if has_carry:
        body.append(f"  assign cout = {current(design.outputs['c4'])};")
    outs = set(design.outputs.values())
    garbage = [current(l) for l in sorted(used) if l not in outs]

    inventory = ", ".join(f"{counts[k]} {k}" for k in ("Peres", "Feynman", "Toffoli", "NOT",
                                                       "TSG", "R2", "MCT") if counts[k])
    ancilla = sum(1 for l in used if l in design.ancilla)
    header = [f"// {note}",
              f"// Gates: {inventory}.  Ancilla inputs: {ancilla}.  Garbage outputs: {len(garbage)}.",
              f"module {name} (",
              "  input  [3:0] a,",
              "  input  [3:0] b,",
              "  input        cin,",
              "  output [3:0] sum" + ("," if has_carry else ""),
              ]
    if has_carry:
        header.append("  output       cout")
    header.append(");")
    decl = []
    for start in range(0, len(wires), 8):
        decl.append("  wire " + ", ".join(wires[start:start + 8]) + ";")
    tail = ["", "  // Garbage outputs (not used): " + ", ".join(garbage), "endmodule", ""]
    return "\n".join(header + decl + [""] + body + tail)


def testbench(modules: Sequence[Tuple[str, bool]]) -> str:
    """Self-checking testbench. `modules` lists (module name, has carry output)."""
    t = ["// Self-checking testbench for the reversible 4-bit adders.",
         "// Part 1 applies a few named cases, convenient for a waveform figure.",
         "// Part 2 applies all 512 input combinations and counts wrong answers.",
         "`timescale 1ns / 1ps",
         "",
         "module tb_rev_adders;",
         "  reg  [3:0] a, b;",
         "  reg        cin;",
         "  reg  [4:0] expected;",
         "  integer    i, total;"]
    for k, (name, carry) in enumerate(modules, 1):
        t.append(f"  wire [3:0] sum{k};" + (f"  wire cout{k};" if carry else ""))
        t.append(f"  integer    errors{k};")
    t.append("")
    for k, (name, carry) in enumerate(modules, 1):
        ports = f".a(a), .b(b), .cin(cin), .sum(sum{k})" + (f", .cout(cout{k})" if carry else "")
        t.append(f"  {name} dut{k} ({ports});")
    t += ["",
          "  task check;",
          "    begin",
          "      expected = a + b + cin;"]
    for k, (name, carry) in enumerate(modules, 1):
        got = f"{{cout{k}, sum{k}}} !== expected" if carry else f"sum{k} !== expected[3:0]"
        t.append(f"      if ({got}) errors{k} = errors{k} + 1;")
    t += ["    end",
          "  endtask",
          "",
          "  initial begin"]
    for k in range(1, len(modules) + 1):
        t.append(f"    errors{k} = 0;")
    t += ["",
          "    // Part 1: named cases",
          "    a = 4'd5;  b = 4'd3; cin = 1'b0; #50; check;   //  5 + 3     =  8",
          "    a = 4'd15; b = 4'd1; cin = 1'b0; #50; check;   // 15 + 1     = 16 (sum 0, carry 1)",
          "    a = 4'd10; b = 4'd5; cin = 1'b0; #50; check;   // 10 + 5     = 15",
          "    a = 4'd9;  b = 4'd9; cin = 1'b1; #50; check;   //  9 + 9 + 1 = 19 (sum 3, carry 1)",
          "",
          "    // Part 2: all 512 input combinations (the whole test ends at 712 ns)",
          "    for (i = 0; i < 512; i = i + 1) begin",
          "      {a, b, cin} = i[8:0];",
          "      #1;",
          "      check;",
          "    end",
          "",
          "    total = 0;"]
    for k, (name, carry) in enumerate(modules, 1):
        what = "sum and carry" if carry else "sum only"
        t.append(f'    $display("{name:<22} ({what:<13}) wrong answers: %0d of 516", errors{k});')
        t.append(f"    total = total + errors{k};")
    t += ['    if (total == 0) $display("ALL TESTS PASSED");',
          '    else            $display("TESTS FAILED: %0d wrong answers", total);',
          "    $finish;",
          "  end",
          "endmodule",
          ""]
    return "\n".join(t)


def gate_testbench() -> str:
    """Self-checking testbench for the gate library.

    The expected outputs come from the Python reference functions in
    adders.py, so this compares the Verilog gates with the Python model.
    """
    from adders import feynman_ref, peres_ref, r2_ref, tsg_ref
    refs = [("rev_not", "a", "p", lambda a: (1 - a,)),
            ("feynman_gate", "ab", "pq", feynman_ref),
            ("toffoli_gate", "abc", "pqr", lambda a, b, c: (a, b, (a & b) ^ c)),
            ("peres_gate", "abc", "pqr", peres_ref),
            ("tsg_gate", "abcd", "pqrs", tsg_ref),
            ("r2_gate", "abcd", "pqrs", r2_ref)]
    t = ["// Self-checking testbench for the reversible gate library.",
         "// Every gate is tried on every input; expected values come from the Python model.",
         "`timescale 1ns / 1ps",
         "",
         "module tb_rev_gates;",
         "  reg  [3:0] x;",
         "  integer    errors, total;"]
    for k, (module, ins, outs, _) in enumerate(refs, 1):
        width = len(outs)
        t.append(f"  wire [{width - 1}:0] y{k};")
        ports = [f".{port}(x[{len(ins) - 1 - j}])" for j, port in enumerate(ins)]
        ports += [f".{port}(y{k}[{width - 1 - j}])" for j, port in enumerate(outs)]
        t.append(f"  {module} g{k} ({', '.join(ports)});")
    t += ["", "  initial begin", "    total = 0;"]
    for k, (module, ins, outs, ref) in enumerate(refs, 1):
        n = len(ins)
        t.append(f"    errors = 0;                                  // {module}")
        for value in range(1 << n):
            bits = [(value >> (n - 1 - j)) & 1 for j in range(n)]
            want = "".join(str(b) for b in ref(*bits))
            given = "".join(str(b) for b in bits).rjust(4, "0")
            t.append(f"    x = 4'b{given}; #1; if (y{k} !== {len(outs)}'b{want}) errors = errors + 1;")
        t.append(f'    $display("{module:<13} wrong answers: %0d of {1 << n}", errors);')
        t.append("    total = total + errors;")
    t += ['    if (total == 0) $display("ALL GATE TESTS PASSED");',
          '    else            $display("GATE TESTS FAILED: %0d wrong answers", total);',
          "    $finish;",
          "  end",
          "endmodule",
          ""]
    return "\n".join(t)


def build_designs() -> List[Tuple[str, Design, List[Block], str]]:
    """The five adder modules: (module name, design, blocks, description)."""
    out = []
    for module, design in (("cla", cla_adder()), ("cla_carry", cla_adder(carry_out=True))):
        tail = "with carry-out" if "c4" in design.outputs else "sum bits only"
        out.append((f"{module}_as_built", design, list(design.blocks),
                    f"Carry look-ahead adder, {tail}, reconstructed from Kumar et al. (IACIS 2024)."))
        result = simplify_design(design)
        out.append((f"{module}_simplified", design.with_circuit(result.circuit, design.name),
                    blocks_from_circuit(result.circuit),
                    f"Carry look-ahead adder, {tail}, after ancilla- and garbage-aware simplification."))
    rca = rca_adder()
    out.append(("rca_peres", rca, list(rca.blocks),
                "Ripple-carry adder with carry-out, two Peres gates per bit."))
    return out


def write_files(folder: str = "verilog") -> Dict[str, str]:
    """Write the three Verilog files into `folder`. Returns {file name: path}."""
    os.makedirs(folder, exist_ok=True)
    designs = build_designs()
    adders = ["// Reversible 4-bit adders built from the gates in rev_gates.v.",
              "// Generated by verilog.py (Phase 6). Do not edit by hand.",
              "`timescale 1ns / 1ps", ""]
    for name, design, blocks, note in designs:
        adders.append(adder_module(name, design, blocks, note))
    contents = {
        "rev_gates.v": LIBRARY,
        "rev_adders.v": "\n".join(adders),
        "tb_rev_adders.v": testbench([(name, "c4" in d.outputs) for name, d, _, _ in designs]),
        "tb_rev_gates.v": gate_testbench(),
    }
    paths = {}
    for file_name, text in contents.items():
        paths[file_name] = os.path.join(folder, file_name)
        with open(paths[file_name], "w", encoding="utf-8") as handle:
            handle.write(text)
    return paths


def simulate(folder: str = "verilog", top: str = "tb_rev_adders"):
    """Compile the files in `folder` and run one testbench with Icarus Verilog.

    Returns the simulator output, or None when iverilog is not installed.
    """
    if shutil.which("iverilog") is None or shutil.which("vvp") is None:
        return None
    sources = [os.path.join(folder, f) for f in ("rev_gates.v", "rev_adders.v", top + ".v")]
    with tempfile.TemporaryDirectory() as scratch:        # keep compiled files out of `folder`
        binary = os.path.join(scratch, top + ".vvp")
        compiled = subprocess.run(["iverilog", "-g2001", "-Wall", "-s", top, "-o", binary] + sources,
                                  capture_output=True, text=True)
        if compiled.returncode != 0:
            return "COMPILE ERROR\n" + compiled.stdout + compiled.stderr
        run = subprocess.run(["vvp", "-n", binary], capture_output=True, text=True)
    return (compiled.stdout + compiled.stderr + run.stdout + run.stderr).strip()


def instance_counts(verilog_text: str, module: str) -> Counter:
    """Count gate instances inside one module of a Verilog file."""
    body = re.search(rf"module {module} \(.*?endmodule", verilog_text, re.S).group(0)
    return Counter(re.findall(r"^\s+(rev_not|feynman_gate|toffoli_gate|peres_gate|tsg_gate|r2_gate)\s",
                              body, re.M))

In [ ]:
%%writefile figures.py
"""Figures for the thesis: circuit diagrams and a simulation waveform (Phase 7).

  gate_library_figure   the four blocks (Feynman, Peres, TSG, R2) as
                        Toffoli-type circuits
  circuit_figure        any adder as a reversible circuit diagram, with its
                        blocks marked along the top
  waveform_figure       inputs and outputs of the Verilog adders for the four
                        named test cases, taken from an Icarus Verilog run

Diagram symbols: filled dot = positive control, open dot = negative control,
circled plus = target. Time runs from left to right.

  depth_figure          Toffoli depth of the Phase 8 designs against word length
  design_figure         a Phase 8 adder design (TSG, Cuccaro, VBE, Draper) as a
                        circuit diagram with its blocks marked

All figures are black on white so that they print well. They need matplotlib;
the waveform also needs the Icarus Verilog simulator (command `iverilog`).

    python figures.py            figures at 300 dpi
    python figures.py --print    figures at 600 dpi, each also as a vector PDF
"""
from __future__ import annotations

import os
import re
import shutil
import subprocess
import tempfile
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

from revcircuit import Circuit, Gate
from adders import Design, cla_adder, feynman, peres, r2, rca_adder, tsg
from simplify import simplify_design
import verilog

DPI = 300            # IEEE asks for at least 600 dpi for line art; make_all(dpi=600) gives that
VECTOR = False       # True: also write a vector PDF next to every PNG

INK = "#000000"
SOFT = "#555555"
BAND = "#e6e6e6"
GATES_IN = {"Feynman": 1, "Peres": 2, "Toffoli": 1, "NOT": 1, "MCT": 1, "TSG": 5, "R2": 3}
SHORT = {"Feynman": "F", "Peres": "P", "Toffoli": "T", "NOT": "N", "MCT": "M", "TSG": "TSG", "R2": "R2"}


def _save(fig, path: str) -> None:
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    if VECTOR:
        fig.savefig(os.path.splitext(path)[0] + ".pdf", bbox_inches="tight", facecolor="white")


def _math(name: str) -> str:
    """'a0' -> '$A_0$', 'c0' -> '$C_0$' for input labels."""
    return f"${name[0].upper()}_{{{name[1:]}}}$"


def _draw_gates(ax, gates: Sequence[Gate], row: Dict[int, int], x0: float = 1.0) -> None:
    for k, g in enumerate(gates):
        x = x0 + k
        ys = [row[l] for l in g.lines]
        ax.plot([x, x], [min(ys), max(ys)], color=INK, lw=1.1, zorder=2)
        for l in g.pos:
            ax.add_patch(Circle((x, row[l]), 0.13, fc=INK, ec=INK, lw=1.0, zorder=3))
        for l in g.neg:
            ax.add_patch(Circle((x, row[l]), 0.15, fc="white", ec=INK, lw=1.1, zorder=3))
        y = row[g.target]
        ax.add_patch(Circle((x, y), 0.26, fc="white", ec=INK, lw=1.1, zorder=3))
        ax.plot([x - 0.26, x + 0.26], [y, y], color=INK, lw=1.1, zorder=4)
        ax.plot([x, x], [y - 0.26, y + 0.26], color=INK, lw=1.1, zorder=4)


def circuit_figure(path: str, design: Design, circuit: Circuit,
                   blocks: Optional[Sequence[Tuple[str, tuple]]] = None,
                   title: str = "") -> str:
    """Draw an adder circuit. Lines that no gate touches are left out."""
    used = set(design.inputs.values()) | set(design.outputs.values())
    for g in circuit:
        used |= g.lines
    lines = sorted(used)
    row = {l: i for i, l in enumerate(lines)}
    n_gates, n_rows = len(circuit), len(lines)
    x_end = n_gates + 1

    fig, ax = plt.subplots(figsize=(0.30 * n_gates + 3.2, 0.33 * n_rows + 1.3), dpi=DPI)

    if blocks:                                         # alternate shading marks the blocks
        k = 0
        for index, (kind, _) in enumerate(blocks):
            width = GATES_IN[kind]
            if index % 2 == 0:
                ax.add_patch(Rectangle((k + 0.5, -0.6), width, n_rows + 0.2, fc=BAND, ec="none", zorder=0))
            ax.text(k + 0.5 + width / 2, -0.95, SHORT[kind], ha="center", va="center",
                    fontsize=7.5, color=INK)
            k += width

    outputs = {l: name for name, l in design.outputs.items()}
    names = design.line_names
    for l in lines:
        y = row[l]
        ax.plot([0.3, x_end - 0.3], [y, y], color=INK, lw=0.7, zorder=1)
        is_input = names[l] in design.inputs
        ax.text(-0.1, y, _math(names[l]) if is_input else "0", ha="right", va="center",
                fontsize=9, color=INK)
        ax.text(-1.5, y, names[l], ha="right", va="center", fontsize=7.5, color=SOFT,
                family="monospace")
        if l in outputs:
            ax.text(x_end + 0.1, y, _math(outputs[l]), ha="left", va="center", fontsize=9,
                    color=INK, fontweight="bold")
    _draw_gates(ax, circuit.gates, row)

    ax.text(-1.5, -0.95, "line", ha="right", va="center", fontsize=7, color=SOFT)
    ax.text(-0.1, -0.95, "input", ha="right", va="center", fontsize=7, color=SOFT)
    ax.text(x_end + 0.1, -0.95, "output", ha="left", va="center", fontsize=7, color=SOFT)
    if blocks:
        kinds = [k for k in ("Peres", "Feynman", "Toffoli", "TSG", "R2", "NOT", "MCT")
                 if any(kind == k for kind, _ in blocks)]
        key = "Blocks:  " + ",  ".join(f"{SHORT[k]} = {k}" for k in kinds if SHORT[k] != k)
        ax.text(0.5, n_rows + 0.15, key, ha="left", va="center", fontsize=7.5, color=SOFT)
    if title:
        ax.set_title(title, fontsize=10, color=INK, pad=6)
    ax.set_xlim(-3.2, x_end + 1.6)
    ax.set_ylim(n_rows + 0.5, -1.4)
    ax.set_aspect("equal")
    ax.axis("off")
    _save(fig, path)
    plt.close(fig)
    return path


def gate_library_figure(path: str) -> str:
    """The four blocks of the adder paper as Toffoli-type circuits."""
    panels = [
        ("Feynman gate", feynman(0, 1), "AB", ["A", "A ⊕ B"]),
        ("Peres gate", peres(0, 1, 2), "ABC", ["A", "A ⊕ B", "AB ⊕ C"]),
        ("TSG gate", tsg(0, 1, 2, 3), "ABCD", ["A", "Q", "QD ⊕ AB ⊕ C", "Q ⊕ D"]),
        ("R2 gate", r2(0, 1, 2, 3), "ABCD", ["A", "B", "C", "D ⊕ A ⊕ B ⊕ C"]),
    ]
    widths = [len(g) + 5.2 for _, g, _, _ in panels]
    fig, axes = plt.subplots(1, 4, figsize=(0.46 * sum(widths), 2.6), dpi=DPI,
                             gridspec_kw={"width_ratios": widths, "wspace": 0.05})
    for ax, (name, gates, ins, outs) in zip(axes, panels):
        n = len(ins)
        row = {l: l for l in range(n)}
        x_end = len(gates) + 1
        for l in range(n):
            ax.plot([0.3, x_end - 0.3], [l, l], color=INK, lw=0.7, zorder=1)
            ax.text(0.1, l, ins[l], ha="right", va="center", fontsize=9, color=INK, style="italic")
            ax.text(x_end - 0.1, l, outs[l], ha="left", va="center", fontsize=8.5, color=INK)
        _draw_gates(ax, gates, row)
        ax.set_title(name, fontsize=9.5, color=INK, pad=4)
        ax.set_xlim(-0.6, x_end + 3.6)
        ax.set_ylim(3.6, -0.7)
        ax.set_aspect("equal")
        ax.axis("off")
    fig.text(0.5, 0.02, "TSG gate:  Q = A'C' ⊕ B'", ha="center", fontsize=8, color=SOFT)
    _save(fig, path)
    plt.close(fig)
    return path


# ---------------------------------------------------------------------------
# Waveform
# ---------------------------------------------------------------------------
WAVE_CASES = [(5, 3, 0), (15, 1, 0), (10, 5, 0), (9, 9, 1)]          # (a, b, cin), 50 ns each
WAVE_DUTS = [("cla_as_built", False), ("cla_simplified", False),
             ("cla_carry_simplified", True), ("rca_peres", True)]


def _wave_testbench() -> str:
    t = ["`timescale 1ns / 1ps", "module tb_waveform;", "  reg [3:0] a, b;", "  reg cin;"]
    for name, carry in WAVE_DUTS:
        t.append(f"  wire [3:0] sum_{name};" + (f" wire cout_{name};" if carry else ""))
        ports = f".a(a), .b(b), .cin(cin), .sum(sum_{name})" + (f", .cout(cout_{name})" if carry else "")
        t.append(f"  {name} dut_{name} ({ports});")
    t += ["  initial begin", '    $dumpfile("wave.vcd");', "    $dumpvars(1, tb_waveform);"]
    for a, b, cin in WAVE_CASES:
        t.append(f"    a = 4'd{a}; b = 4'd{b}; cin = 1'b{cin}; #50;")
    t += ["    $finish;", "  end", "endmodule", ""]
    return "\n".join(t)


def _read_vcd(text: str) -> Dict[str, List[Tuple[int, int]]]:
    """Value changes per signal: {name: [(time in ns, value), ...]}."""
    names, changes, now = {}, {}, 0
    scale = 1.0
    m = re.search(r"\$timescale\s+(\d+)\s*(\w+)\s+\$end", text)
    if m:
        scale = int(m.group(1)) * {"s": 1e9, "ms": 1e6, "us": 1e3, "ns": 1.0, "ps": 1e-3, "fs": 1e-6}[m.group(2)]
    for line in text.splitlines():
        line = line.strip()
        var = re.match(r"\$var\s+\w+\s+\d+\s+(\S+)\s+(\S+)", line)
        if var:
            names[var.group(1)] = var.group(2)
            changes[var.group(2)] = []
        elif line.startswith("#"):
            now = int(round(int(line[1:]) * scale))
        elif line.startswith("b"):
            bits, ident = line[1:].split()
            if ident in names and set(bits) <= {"0", "1"}:
                changes[names[ident]].append((now, int(bits, 2)))
        elif line and line[0] in "01" and line[1:] in names:
            changes[names[line[1:]]].append((now, int(line[0])))
    return changes


def waveform_figure(path: str, folder: str = "verilog") -> Optional[str]:
    """Simulate the four named cases and draw the waveform. None if no simulator."""
    if shutil.which("iverilog") is None or shutil.which("vvp") is None:
        return None
    with tempfile.TemporaryDirectory() as scratch:
        tb = os.path.join(scratch, "tb_waveform.v")
        with open(tb, "w", encoding="utf-8") as handle:
            handle.write(_wave_testbench())
        sources = [os.path.abspath(os.path.join(folder, f)) for f in ("rev_gates.v", "rev_adders.v")]
        subprocess.run(["iverilog", "-g2001", "-s", "tb_waveform", "-o", "wave.vvp", tb] + sources,
                       cwd=scratch, check=True, capture_output=True)
        subprocess.run(["vvp", "-n", "wave.vvp"], cwd=scratch, check=True, capture_output=True)
        with open(os.path.join(scratch, "wave.vcd"), encoding="utf-8") as handle:
            data = _read_vcd(handle.read())

    end = 50 * len(WAVE_CASES)
    rows = [("a[3:0]", "a", True), ("b[3:0]", "b", True), ("cin", "cin", False)]
    for name, carry in WAVE_DUTS:
        rows.append((f"sum[3:0]  {name}", f"sum_{name}", True))
        if carry:
            rows.append((f"cout  {name}", f"cout_{name}", False))

    fig, ax = plt.subplots(figsize=(10, 0.46 * len(rows) + 1.2), dpi=DPI)
    for k in range(len(WAVE_CASES) + 1):
        ax.plot([50 * k, 50 * k], [-0.7, len(rows) - 0.3], color="#bbbbbb", lw=0.6, ls=(0, (3, 3)), zorder=0)
    for r, (label, key, bus) in enumerate(rows):
        seq = data[key] + [(end, None)]
        ax.text(-4, r, label, ha="right", va="center", fontsize=8.5, color=INK, family="monospace")
        for (t0, value), (t1, _) in zip(seq, seq[1:]):
            if t1 <= t0:
                continue
            if bus:                                    # a bus is a closed box holding its value
                s = 1.2
                ax.plot([t0, t0 + s, t1 - s, t1, t1 - s, t0 + s, t0],
                        [r, r - 0.3, r - 0.3, r, r + 0.3, r + 0.3, r], color=INK, lw=1.0)
                ax.text((t0 + t1) / 2, r, str(value), ha="center", va="center", fontsize=9, color=INK)
            else:                                      # one bit: high is drawn above low
                y = r - 0.3 if value else r + 0.3
                ax.plot([t0, t1], [y, y], color=INK, lw=1.2)
                ax.text((t0 + t1) / 2, r, str(value), ha="center", va="center", fontsize=8, color=SOFT)
        if not bus:
            for (t0, v0), (t1, v1) in zip(data[key], data[key][1:]):
                if v0 != v1:
                    ax.plot([t1, t1], [r - 0.3, r + 0.3], color=INK, lw=1.2)
    for k, (a, b, cin) in enumerate(WAVE_CASES):
        ax.text(50 * k + 25, -1.05, f"{a} + {b} + {cin} = {a + b + cin}", ha="center", va="center",
                fontsize=8.5, color=INK)
    ax.set_xlim(-2, end + 2)
    ax.set_ylim(len(rows) - 0.2, -1.5)
    ax.set_xticks([50 * k for k in range(len(WAVE_CASES) + 1)])
    ax.set_xlabel("Time (ns)", fontsize=9, color=INK)
    ax.set_yticks([])
    for side in ("left", "right", "top"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="x", labelsize=8.5, colors=INK)
    _save(fig, path)
    plt.close(fig)
    return path


def depth_figure(path: str, widths: Sequence[int] = (4, 8, 16, 32)) -> str:
    """Toffoli depth of five adder designs as built, against the word length."""
    from nadders import DESIGNS, toffoli_depth
    series = [("vbe", "VBE ripple-carry", "s", "-"),
              ("cuccaro", "Cuccaro ripple-carry", "^", "-"),
              ("cla4_blocks", "CLA blocks (Kumar style)", "D", "--"),
              ("peres_rca", "Peres ripple-carry", "o", "-"),
              ("draper_qcla", "Draper carry look-ahead", "v", "-")]
    fig, ax = plt.subplots(figsize=(6.4, 4.2), dpi=DPI)
    for key, label, marker, style in series:
        values = [toffoli_depth(DESIGNS[key](n).circuit) for n in widths]
        ax.plot(widths, values, color=INK, lw=1.3, ls=style, marker=marker, ms=6,
                mfc="white", mec=INK, mew=1.2, zorder=3)
        ax.annotate(f"{label}  ({values[-1]})", (widths[-1], values[-1]), xytext=(9, 0),
                    textcoords="offset points", va="center", fontsize=8.5, color=INK)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log", base=2)
    ax.set_xticks(list(widths))
    ax.set_xticklabels([str(n) for n in widths])
    ax.set_yticks([4, 8, 16, 32, 64, 128])
    ax.set_yticklabels(["4", "8", "16", "32", "64", "128"])
    ax.set_xlim(widths[0] * 0.85, widths[-1] * 1.12)
    ax.set_ylim(3.5, 140)
    ax.set_xlabel("Word length n (bits)", fontsize=9.5, color=INK)
    ax.set_ylabel("Toffoli depth", fontsize=9.5, color=INK)
    ax.grid(True, which="major", color="#dddddd", lw=0.6, zorder=0)
    ax.minorticks_off()
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(labelsize=9, colors=INK)
    _save(fig, path)
    plt.close(fig)
    return path


# ---------------------------------------------------------------------------
# The adder designs of Phase 8 (nadders.py)
# ---------------------------------------------------------------------------
def adder_bands(key: str, n: int) -> List[Tuple[str, int]]:
    """(label, number of gates) for each block of a Phase 8 design as built."""
    if key == "tsg_rca":
        return [("TSG", 5)] * n
    if key == "peres_rca":
        return [("P", 2)] * (2 * n)
    if key == "cuccaro":
        return [("MAJ", 3)] * n + [("", 1)] + [("UMA", 3)] * n
    if key == "vbe":
        inv = "CARRY$^{-1}$"
        return [("CARRY", 3)] * n + [("", 1), ("SUM", 2)] + [(inv, 3), ("SUM", 2)] * (n - 1)
    if key == "draper_qcla":
        log = n.bit_length() - 1
        p_round = sum((n >> t) - 1 for t in range(1, log))
        g_round = sum(n >> t for t in range(1, log + 1))
        top = 0
        while 3 * (1 << (top + 1)) <= 2 * n:
            top += 1
        c_round = sum((n - (1 << (t - 1))) // (1 << t) for t in range(top, 0, -1))
        bands = [("generate", n), ("propagate", n - 1), ("P", p_round), ("G", g_round),
                 ("C", c_round), ("P$^{-1}$", p_round), ("sum", n + 1), ("restore", n - 1)]
        return [(label, width) for label, width in bands if width]
    raise ValueError(key)


def design_figure(path: str, design, circuit: Optional[Circuit] = None,
                  bands: Optional[Sequence[Tuple[str, int]]] = None) -> str:
    """Draw a Phase 8 adder design (an AdderDesign of nadders.py).

    Left: line name and input (an operand bit, or 0 for a line that starts at 0).
    Right: S for a sum bit, C for the carry-out, the operand name for a restored
    line and 0 for an ancilla line that a clean design returns to 0. A line
    without a label on the right is a garbage output.
    """
    circuit = design.circuit if circuit is None else circuit
    n_rows, n_gates = design.n, len(circuit)
    row = {l: l for l in range(n_rows)}
    x_end = n_gates + 1
    fig, ax = plt.subplots(figsize=(0.30 * n_gates + 3.2, 0.33 * n_rows + 1.3), dpi=DPI)

    if bands:
        k = 0
        for index, (label, width) in enumerate(bands):
            if index % 2 == 0:
                ax.add_patch(Rectangle((k + 0.5, -0.6), width, n_rows + 0.2, fc=BAND, ec="none", zorder=0))
            ax.text(k + 0.5 + width / 2, -0.95, label, ha="center", va="center", fontsize=7, color=INK)
            k += width

    line_of_input = {l: name for name, l in design.inputs.items()}
    out_label: Dict[int, Tuple[str, bool]] = {}
    for l, name in design.restored.items():
        out_label[l] = (_math(name), False)
    if design.style == "clean":
        for l in design.ancilla:
            out_label[l] = ("0", False)
    for i, l in enumerate(design.sum_lines):
        out_label[l] = (f"$S_{{{i}}}$", True)
    if design.carry_line is not None:
        out_label[design.carry_line] = (f"$C_{{{design.bits}}}$", True)

    for l in range(n_rows):
        ax.plot([0.3, x_end - 0.3], [l, l], color=INK, lw=0.7, zorder=1)
        ax.text(-0.1, l, _math(line_of_input[l]) if l in line_of_input else "0", ha="right",
                va="center", fontsize=9, color=INK)
        ax.text(-1.5, l, design.line_names[l], ha="right", va="center", fontsize=7.5, color=SOFT,
                family="monospace")
        if l in out_label:
            text, bold = out_label[l]
            ax.text(x_end + 0.1, l, text, ha="left", va="center", fontsize=9, color=INK,
                    fontweight="bold" if bold else "normal")
    _draw_gates(ax, circuit.gates, row)

    ax.text(-1.5, -0.95, "line", ha="right", va="center", fontsize=7, color=SOFT)
    ax.text(-0.1, -0.95, "input", ha="right", va="center", fontsize=7, color=SOFT)
    ax.text(x_end + 0.1, -0.95, "output", ha="left", va="center", fontsize=7, color=SOFT)
    ax.set_xlim(-3.2, x_end + 1.6)
    ax.set_ylim(n_rows - 0.3, -1.4)
    ax.set_aspect("equal")
    ax.axis("off")
    _save(fig, path)
    plt.close(fig)
    return path


def design_figures(folder: str = "figures", bits: int = 4) -> Dict[str, str]:
    """The Phase 8 designs that the earlier figures do not show, as built, and the
    two that simplification changes into a new circuit, after all steps."""
    from nadders import DESIGNS
    from simplify import simplify
    out = {}
    for key in ("tsg_rca", "cuccaro", "vbe", "draper_qcla"):
        design = DESIGNS[key](bits)
        name = key.replace("_qcla", "")
        out[name] = design_figure(os.path.join(folder, f"fig_{name}.png"), design,
                                  bands=adder_bands(key, bits))
        if key in ("cuccaro", "vbe"):
            result = simplify(design.circuit, list(design.inputs.values()), design.ancilla, design.required)
            out[name + "_simplified"] = design_figure(
                os.path.join(folder, f"fig_{name}_simplified.png"), design, result.circuit)
    try:                                                   # Phase 11: the hybrid adder, at 8 bits
        from hybrid import hybrid_bands, hybrid_cla
        out["hybrid"] = design_figure(os.path.join(folder, "fig_hybrid.png"), hybrid_cla(8),
                                      bands=hybrid_bands(8))
    except ImportError:
        pass
    return out


def make_all(folder: str = "figures", verilog_folder: str = "verilog", dpi: int = 300,
             vector: bool = False) -> Dict[str, Optional[str]]:
    """Write every figure as a PNG file into `folder`. Returns {name: path}.

    dpi=600 meets the IEEE requirement for line art; vector=True also writes
    a PDF version of every figure.
    """
    global DPI, VECTOR
    DPI, VECTOR = dpi, vector
    os.makedirs(folder, exist_ok=True)
    verilog.write_files(verilog_folder)
    out: Dict[str, Optional[str]] = {}
    out["gate_library"] = gate_library_figure(os.path.join(folder, "fig_gate_library.png"))
    for key, design in (("cla", cla_adder()), ("cla_carry", cla_adder(carry_out=True))):
        out[f"{key}_as_built"] = circuit_figure(
            os.path.join(folder, f"fig_{key}_as_built.png"), design, design.circuit, design.blocks)
        result = simplify_design(design)
        out[f"{key}_simplified"] = circuit_figure(
            os.path.join(folder, f"fig_{key}_simplified.png"), design, result.circuit,
            verilog.blocks_from_circuit(result.circuit))
    rca = rca_adder()
    out["rca_peres"] = circuit_figure(os.path.join(folder, "fig_rca_peres.png"), rca, rca.circuit, rca.blocks)
    out["waveform"] = waveform_figure(os.path.join(folder, "fig_waveform.png"), verilog_folder)
    try:
        out["toffoli_depth"] = depth_figure(os.path.join(folder, "fig_toffoli_depth.png"))
        out.update(design_figures(folder))
    except ImportError:                                   # nadders.py (Phase 8) not present
        out["toffoli_depth"] = None
    return out


if __name__ == "__main__":
    import sys
    options = dict(dpi=600, vector=True) if "--print" in sys.argv else {}
    for name, path in make_all(**options).items():
        print(f"{name:<22}{path or 'SKIPPED (no Verilog simulator found)'}")

## Step 2: New files of this phase

In [ ]:
%%writefile arules.py
"""Local rewrite rules for circuits with ancilla and garbage lines (Phase 11).

The complete rule set RC of Feng and Li preserves the function of a circuit
on every line and for every input. A practical circuit has more freedom:
ancilla lines start at a known constant and garbage lines are never read.
Feng and Li leave this case open. This module states five rules for it. Each
rule rewrites ONE gate h, and its side condition looks only at the gates
before h (rules A1, A3, A4, A5) or after h (rule A2). No global simulation of
the circuit is needed, which is the difference from the state-aware analysis
of simplify.py.

Notation: a gate is G[P, N, t]. A line is "fresh" while no gate has targeted
it yet; a fresh ancilla holds its constant. "Window": the stretch of the
circuit between two gates.

A1  Constant control.  If h has a control on a fresh ancilla r with constant
    v, the control is either always satisfied (drop it) or never satisfied
    (delete h).

A2  Dead gate.  If the target of h is not a primary output and no later gate
    uses it as a control, delete h.

A3  Product implication.  Let a fresh 0-ancilla c be written by g0 = G[P0, N0,
    c], and let c be untouched between g0 and h. After g0, c = 1 implies every
    literal of g0 whose line is also untouched in that window. If h has c as a
    positive control:
      A3a  a literal of h contradicts an implied literal  -> delete h,
      A3b  a literal of h equals an implied literal       -> drop it from h.
    If no line of g0 was touched in the window, c equals the product exactly,
    and for a negative control on c:
      A3c  the other literals of h include all literals of g0 -> delete h,
      A3d  the other literals of h contradict g0               -> drop c from h.

A4  Literal substitution.  If c equals one literal x or not-x exactly (g0 has
    one control and the window condition of A3 holds), replace the control c
    of h by that literal. The copy gate g0 can then become dead (A2).

A5  Linear substitution.  If c was written, from 0, only by CNOT and NOT gates
    whose control lines S are untouched in the window, then c = k xor (xor of
    S). A CNOT controlled by c alone is replaced by one CNOT from each line of
    S (and a NOT when the constant k requires it).

Soundness. In every case the firing condition of h is unchanged on every
valid input (ancilla lines at their constants), so every line keeps its value
after h; A2 changes only a line that is never read again and is not an output.
Each rule therefore preserves the primary outputs for all valid inputs (design
equivalence). The rules are not claimed to be complete.

ancilla_optimize() alternates these rules with the RC optimizer of Phase 3,
which moves gates and so creates new instances, and keeps a round only if it
lowers (quantum cost, gate count, ancilla lines used). The result is checked
exactly against the original on every valid input.
"""
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, Dict, FrozenSet, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X
from optimizer import circuit_cost, optimize, quantum_cost
from simplify import _Care


@dataclass
class _Def:
    """What is known about a line from the gates before the current point."""
    kind: str                                  # "const", "prod" or "xor"
    value: int = 0                             # const: the constant; xor: the constant term
    pos: FrozenSet[int] = frozenset()          # prod: literals implied by c = 1
    neg: FrozenSet[int] = frozenset()
    exact: bool = True                         # prod: c equals the product exactly
    support: FrozenSet[int] = frozenset()      # xor: the lines S


@dataclass
class ARuleResult:
    original: Circuit
    circuit: Circuit
    counts: Counter = field(default_factory=Counter)
    rounds: int = 0


ALL_RULES = ("A1", "A2", "A3", "A4", "A5")


def _forward(gates: List[Gate], ancilla: Sequence[int], ones: Sequence[int],
             substitute: bool, counts: Counter, use: Sequence[str] = ALL_RULES) -> List[Gate]:
    """One pass of rules A1, A3 (and A4, A5 when `substitute`) from left to right."""
    defs: Dict[int, Optional[_Def]] = {l: _Def("const", 1 if l in ones else 0) for l in ancilla}
    out: List[Gate] = []

    def rewrite(h: Gate) -> Optional[List[Gate]]:
        """Apply the rules to h. Returns the replacement gates ([] = deleted)."""
        changed = True
        while changed:
            changed = False
            for r in sorted(h.controls):
                d = defs.get(r)
                if d is None:
                    continue
                positive = r in h.pos
                others_pos, others_neg = h.pos - {r}, h.neg - {r}
                if d.kind == "const" and "A1" in use:                       # A1
                    if (d.value == 1) == positive:
                        h = G(others_pos, others_neg, h.target)
                        counts["A1 constant control dropped"] += 1
                        changed = True
                        break
                    counts["A1 gate never fires"] += 1
                    return []
                if d.kind == "prod" and "A3" in use:
                    if positive:
                        if (d.pos & others_neg) or (d.neg & others_pos):    # A3a
                            counts["A3a gate never fires"] += 1
                            return []
                        if (d.pos & others_pos) or (d.neg & others_neg):    # A3b
                            h = G(h.pos - d.pos | {r}, h.neg - d.neg, h.target)
                            counts["A3b implied control dropped"] += 1
                            changed = True
                            break
                    elif d.exact:
                        if d.pos <= others_pos and d.neg <= others_neg:      # A3c
                            counts["A3c gate never fires"] += 1
                            return []
                        if (d.pos & others_neg) or (d.neg & others_pos):    # A3d
                            h = G(others_pos, others_neg, h.target)
                            counts["A3d control dropped"] += 1
                            changed = True
                            break
                if d.kind == "prod" and "A4" in use:
                    if substitute and d.exact and len(d.pos) + len(d.neg) == 1:   # A4
                        (x,) = tuple(d.pos | d.neg)
                        same = (x in d.pos) == positive          # literal polarity on x
                        if x in h.controls or x == h.target:
                            continue
                        new_pos = others_pos | ({x} if same else set())
                        new_neg = others_neg | (set() if same else {x})
                        h = G(new_pos, new_neg, h.target)
                        counts["A4 literal substituted"] += 1
                        changed = True
                        break
                if (d.kind == "xor" and "A5" in use and substitute and len(h.controls) == 1
                        and h.target not in d.support and len(d.support) >= 1):     # A5
                    flip = d.value ^ (0 if positive else 1)
                    repl = [CNOT(s, h.target) for s in sorted(d.support)]
                    if flip:
                        repl.append(X(h.target))
                    counts["A5 linear substitution"] += 1
                    return repl
        return [h]

    def update(h: Gate) -> None:
        t = h.target
        old = defs.get(t)
        for c, d in list(defs.items()):                 # lines whose knowledge depended on t
            if d is None or c == t:
                continue
            if d.kind == "prod" and (t in d.pos or t in d.neg):
                defs[c] = _Def("prod", 0, d.pos - {t}, d.neg - {t}, False)
            elif d.kind == "xor" and t in d.support:
                defs[c] = None
        new: Optional[_Def] = None
        if old is not None:
            if not h.controls:                                           # NOT gate
                if old.kind == "const":
                    new = _Def("const", 1 - old.value)
                elif old.kind == "xor":
                    new = _Def("xor", 1 - old.value, support=old.support)
            elif old.kind == "const" and old.value == 0 and len(h.controls) >= 1:
                new = _Def("prod", 0, frozenset(h.pos), frozenset(h.neg), True)
            elif old.kind == "const" and len(h.controls) == 1:            # from constant 1
                (y,) = tuple(h.controls)
                new = _Def("xor", 1 ^ (0 if y in h.pos else 1), support=frozenset({y}))
            if len(h.controls) == 1 and old.kind in ("prod", "xor"):
                (y,) = tuple(h.controls)
                if old.kind == "prod" and old.exact and len(old.pos) + len(old.neg) == 1:
                    (x,) = tuple(old.pos | old.neg)
                    old = _Def("xor", 0 if x in old.pos else 1, support=frozenset({x}))
                if old.kind == "xor":
                    support = old.support ^ {y}
                    value = old.value ^ (0 if y in h.pos else 1)
                    new = _Def("xor", value, support=frozenset(support)) if support else _Def("const", value)
        defs[t] = new

    for h in gates:
        for g in rewrite(h):
            out.append(g)
            update(g)
    return out


def _dead(gates: List[Gate], outputs: Sequence[int], counts: Counter) -> List[Gate]:
    """Rule A2 from right to left."""
    read = set()
    outs = set(outputs)
    kept = []
    for g in reversed(gates):
        if g.target not in outs and g.target not in read:
            counts["A2 dead gate removed"] += 1
            continue
        kept.append(g)
        read |= g.controls
    return kept[::-1]


def ancilla_optimize(circuit: Circuit, inputs: Sequence[int], ancilla: Sequence[int],
                     outputs: Sequence[int], ancilla_ones: Sequence[int] = (),
                     cost: Callable[[Gate], int] = quantum_cost, use_rc: bool = True,
                     check: bool = True, max_rounds: int = 30,
                     use: Sequence[str] = ALL_RULES) -> ARuleResult:
    """Rules A1 to A5 (those listed in `use`) together with the RC optimizer.

    With use_rc the RC optimizer runs first and after every pass of the rules,
    so the result is never worse than the RC optimizer alone.
    """
    ancilla, ones = list(ancilla), set(ancilla_ones)
    anc_set = set(ancilla)

    def score(gates):
        touched = set()
        for g in gates:
            touched |= g.lines
        return (sum(cost(g) for g in gates), len(gates), len(touched & anc_set))

    gates = list(circuit.gates)
    counts: Counter = Counter()
    if use_rc:
        first = optimize(circuit, cost=cost)
        if score(first.circuit.gates) < score(gates):
            gates = list(first.circuit.gates)
            counts["RC rewrites"] += first.rewrites
    rounds = 0
    while rounds < max_rounds:
        rounds += 1
        best, best_counts = None, None
        for substitute in (False, True):
            trial_counts: Counter = Counter()
            trial = _forward(gates, ancilla, ones, substitute, trial_counts, use)
            if "A2" in use:
                trial = _dead(trial, outputs, trial_counts)
            if use_rc:
                result = optimize(Circuit(circuit.n, trial), cost=cost)
                if result.rewrites:
                    trial_counts["RC rewrites"] += result.rewrites
                trial = list(result.circuit.gates)
            if best is None or score(trial) < score(best):
                best, best_counts = trial, trial_counts
        if score(best) >= score(gates):
            break
        gates = best
        counts.update(best_counts)
    out = Circuit(circuit.n, gates)
    if check:
        care = _Care(circuit.n, inputs, ancilla, outputs, cost, "auto", ancilla_ones)
        if care.signature(gates) != care.signature(list(circuit.gates)):
            raise AssertionError("the ancilla rules changed a primary output")
    return ARuleResult(circuit, out, counts, rounds)

In [ ]:
%%writefile hybrid.py
"""A hybrid carry look-ahead adder (Phase 11).

The design keeps what is cheap in the adder of Kumar et al. and replaces what
is slow:

  * Generate and propagate come from one Peres gate per bit, as in the PGA
    block of Kumar et al.: Peres(A_i, B_i, 0) leaves P_i = A_i xor B_i on the
    B line and G_i = A_i B_i on a fresh line.
  * The carry input is folded into the first generate bit with one Toffoli
    gate: G_0 := G_0 xor P_0 C_0. Every carry is then a group generate.
  * The carries come from the logarithmic-depth prefix network of Draper et
    al. (P-rounds, G-rounds and C-rounds), in place on the generate lines.
  * Unlike the Draper adder, nothing is uncomputed: the design is in the
    garbage style of Kumar et al., where only the sum and the carry-out are
    specified. This removes the inverse P-rounds and the restoring gates.
  * The sum is formed in place on the B lines: S_i = P_i xor C_i.

Lines: A (n, garbage), B (n, sum), carry input c0, generate lines z1..zn
(z_n is the carry-out, the others garbage) and the propagate ancillas of the
prefix network.
"""
from __future__ import annotations

from typing import List

from revcircuit import CNOT, Circuit, Gate, TOF
from adders import peres
from nadders import AdderDesign, _Lines, _interleaved


def hybrid_cla(n: int) -> AdderDesign:
    if n < 2:
        raise ValueError("n must be at least 2")
    log = n.bit_length() - 1                                  # floor(log2 n)
    L = _Lines()
    L.add("c0", *[x for i in range(n) for x in (f"a{i}", f"b{i}")], *[f"z{i}" for i in range(1, n + 1)])
    for t in range(1, log):
        L.add(*[f"p{t}_{m}" for m in range(1, n >> t)])

    def P(t, m):
        return L[f"b{m}"] if t == 0 else L[f"p{t}_{m}"]

    def Z(j):
        return L[f"z{j}"]

    gates: List[Gate] = []
    for i in range(n):                                        # PGA: Peres per bit
        gates += peres(L[f"a{i}"], L[f"b{i}"], Z(i + 1))
    gates.append(TOF(L["b0"], L["c0"], Z(1)))                 # fold the carry input into G_0
    rounds = {"P": 0, "G": 0, "C": 0}
    for t in range(1, log):                                   # P-rounds
        for m in range(1, n >> t):
            gates.append(TOF(P(t - 1, 2 * m), P(t - 1, 2 * m + 1), P(t, m)))
            rounds["P"] += 1
    for t in range(1, log + 1):                               # G-rounds
        for m in range(0, n >> t):
            gates.append(TOF(Z((m << t) + (1 << (t - 1))), P(t - 1, 2 * m + 1), Z((m << t) + (1 << t))))
            rounds["G"] += 1
    top = 0
    while 3 * (1 << (top + 1)) <= 2 * n:                      # floor(log2(2n/3))
        top += 1
    for t in range(top, 0, -1):                               # C-rounds
        for m in range(1, (n - (1 << (t - 1))) // (1 << t) + 1):
            gates.append(TOF(Z(m << t), P(t - 1, 2 * m), Z((m << t) + (1 << (t - 1)))))
            rounds["C"] += 1
    gates.append(CNOT(L["c0"], L["b0"]))                      # S_0 = P_0 xor C_0
    for i in range(1, n):
        gates.append(CNOT(Z(i), L[f"b{i}"]))                  # S_i = P_i xor C_i
    inputs = _interleaved(L, n, "c0")
    ancilla = [i for i, name in enumerate(L.names) if name not in inputs]
    design = AdderDesign(f"Hybrid CLA, {n} bits", "garbage", n, Circuit(len(L.names), gates), L.names,
                         inputs, ancilla, [L[f"b{i}"] for i in range(n)], Z(n), L["c0"], {}, [])
    design.rounds = rounds
    return design


def hybrid_bands(n: int):
    """(label, gates) per stage, for figures."""
    d = hybrid_cla(n)
    r = d.rounds
    bands = [("PGA", 2)] * n + [("cin", 1), ("P", r["P"]), ("G", r["G"]), ("C", r["C"]), ("sum", n)]
    return [(label, width) for label, width in bands if width]

In [ ]:
%%writefile verify_phase11.py
"""Phase 11: rewrite rules for ancilla and garbage lines, and a hybrid adder.

Run:  python verify_phase11.py      (about one minute)

  A. Soundness of rules A1 to A5 on random circuits, checked with the plain
     one-input-at-a-time simulator (independent of the rule code).
  B. The six adders: RC rules alone, RC with rules A1-A5 (local), and the
     global simplifier of Phase 5, at 4 to 32 bits. Exact proofs.
  C. Which rule matters: each rule left out in turn, 32-bit adders.
  D. RevLib circuits: RC alone, RC with A1-A5, global simplifier.
  E. The hybrid carry look-ahead adder: exact proofs at 2 to 32 bits,
     resources, and the comparison with the six designs after all steps,
     with the designs that are Pareto-optimal.
"""
from __future__ import annotations

import random
import sys
import time

from revcircuit import Circuit, G
from optimizer import circuit_cost, optimize
from simplify import simplify
from arules import ALL_RULES, ancilla_optimize
from nadders import DESIGNS, prove_adder, random_test, t_count, toffoli_depth, used_ancilla
from hybrid import hybrid_cla
from adders import depth
import revlib

WIDTHS = (4, 8, 16, 32)
failures = 0


def report(label: str, ok: bool, detail: str = "") -> None:
    global failures
    failures += 0 if ok else 1
    print(f"  {label:<60}{'PASS' if ok else 'FAIL'}  {detail}")


def pct(a, b):
    return f"{100.0 * (a - b) / a:.1f}" if a else "0.0"


def part_a() -> None:
    print("A. Soundness of rules A1-A5 on random circuits")
    rng = random.Random(7)
    wrong = changed = 0
    trials = 800
    for _ in range(trials):
        n = rng.randint(4, 8)
        lines = list(range(n))
        rng.shuffle(lines)
        k = rng.randint(1, n - 1)
        inputs, ancilla = sorted(lines[:k]), sorted(lines[k:])
        ones = [l for l in ancilla if rng.random() < 0.3]
        outputs = sorted(rng.sample(range(n), rng.randint(1, n)))
        gates = []
        for _ in range(rng.randint(5, 25)):
            t = rng.randrange(n)
            others = [l for l in range(n) if l != t]
            ctrl = rng.sample(others, min(rng.choice([0, 1, 1, 2, 2, 2, 3]), len(others)))
            pos = [c for c in ctrl if rng.random() < 0.7]
            gates.append(G(pos, [c for c in ctrl if c not in pos], t))
        c = Circuit(n, gates)
        use = tuple(r for r in ALL_RULES if rng.random() < 0.85) or ALL_RULES
        r = ancilla_optimize(c, inputs, ancilla, outputs, ones, use_rc=rng.random() < 0.5,
                             check=False, use=use)
        changed += r.circuit.gates != c.gates
        base = sum(1 << l for l in ones)
        mask = sum(1 << l for l in outputs)
        for bits in range(1 << k):
            s = base | sum(((bits >> i) & 1) << l for i, l in enumerate(inputs))
            if (c.apply(s) ^ r.circuit.apply(s)) & mask:
                wrong += 1
                break
    report(f"{trials} random circuits, primary outputs unchanged", wrong == 0,
           f"{changed} circuits rewritten, {wrong} wrong")
    print()


def part_b() -> dict:
    print("B. Adders: RC alone, RC + rules A1-A5 (local), global simplifier (quantum cost)")
    print(f"  {'Design':<26}{'Bits':>5}{'As built':>10}{'RC':>7}{'RC+A':>7}{'Global':>8}  Result")
    totals = [0, 0, 0, 0]
    out = {}
    for key, make in DESIGNS.items():
        for n in WIDTHS:
            d = make(n)
            args = (d.circuit, list(d.inputs.values()), d.ancilla, d.required)
            rc = optimize(d.circuit).circuit
            loc = ancilla_optimize(*args)
            glob = simplify(*args).circuit
            ok = prove_adder(d.with_circuit(loc.circuit)) and random_test(d.with_circuit(loc.circuit)) == 0
            v = [circuit_cost(x) for x in (d.circuit, rc, loc.circuit, glob)]
            totals = [a + b for a, b in zip(totals, v)]
            out[key, n] = loc
            print(f"  {d.name.rsplit(',', 1)[0]:<26}{n:>5}{v[0]:>10}{v[1]:>7}{v[2]:>7}{v[3]:>8}  {'PASS' if ok else 'FAIL'}")
            if not ok:
                global failures
                failures += 1
    saved_local, saved_global = totals[0] - totals[2], totals[0] - totals[3]
    print(f"  {'Total':<31}{totals[0]:>10}{totals[1]:>7}{totals[2]:>7}{totals[3]:>8}")
    print(f"  Saving: RC {totals[0] - totals[1]}, RC+A {saved_local}, global {saved_global}. "
          f"The local rules reach {100.0 * saved_local / saved_global:.1f}% of the global saving.")
    print("  Rule uses at 32 bits:")
    for key in DESIGNS:
        counts = out[key, 32].counts
        text = ", ".join(f"{k} x{v}" for k, v in sorted(counts.items())) or "none"
        print(f"    {key:<12} {text}")
    print()
    return out


def part_c() -> None:
    print("C. Each rule left out in turn, 32-bit adders (quantum cost with RC + the other rules)")
    print(f"  {'Design':<14}{'all':>6}" + "".join(f"{'no ' + r:>7}" for r in ALL_RULES))
    for key, make in DESIGNS.items():
        d = make(32)
        args = (d.circuit, list(d.inputs.values()), d.ancilla, d.required)
        full = circuit_cost(ancilla_optimize(*args).circuit)
        row = [circuit_cost(ancilla_optimize(*args, use=tuple(x for x in ALL_RULES if x != r)).circuit)
               for r in ALL_RULES]
        print(f"  {key:<14}{full:>6}" + "".join(f"{v:>7}" for v in row))
    print()


def part_d() -> None:
    print("D. RevLib circuits with constant inputs (RevLib cost table)")
    try:
        folder = revlib.fetch("revlib")
    except Exception as error:
        print(f"  SKIPPED: the RevLib files could not be obtained ({type(error).__name__}).\n")
        return
    totals = [0, 0, 0, 0]
    count = 0
    for b in revlib.load_all(folder):
        if b.outputs is None:
            continue
        cost = revlib.revlib_cost(b.circuit.n)
        rc = optimize(b.circuit, cost=cost).circuit
        loc = ancilla_optimize(b.circuit, b.inputs, b.ancilla, b.outputs, b.ancilla_ones, cost=cost).circuit
        glob = simplify(b.circuit, b.inputs, b.ancilla, b.outputs, cost=cost, ancilla_ones=b.ancilla_ones).circuit
        v = [circuit_cost(x, cost) for x in (b.circuit, rc, loc, glob)]
        totals = [a + c for a, c in zip(totals, v)]
        count += 1
    report(f"{count} circuits: as given / RC / RC+A / global", totals[2] <= totals[1],
           " / ".join(str(x) for x in totals))
    print()


def resources(d, c):
    used = set(d.inputs.values())
    for g in c:
        used |= g.lines
    garbage = 0 if d.style == "clean" else len(used) - (d.bits + 1)
    return dict(cost=circuit_cost(c), T=t_count(c), tdepth=toffoli_depth(c), depth=depth(c),
                anc=used_ancilla(d, c), lines=len(used), garbage=garbage, gates=len(c))


def part_e() -> None:
    print("E. Hybrid carry look-ahead adder (Peres PGA blocks + prefix carries, garbage style)")
    ok = all(prove_adder(hybrid_cla(n)) and random_test(hybrid_cla(n)) == 0
             for n in (2, 3, 4, 5, 6, 7, 8, 12, 16, 32))
    report("proved correct for every input, n = 2-8, 12, 16, 32", ok)
    print(f"  {'Bits':>5}{'Lines':>7}{'Gates':>7}{'Cost':>7}{'T-count':>9}{'Ancilla':>9}{'Garbage':>9}"
          f"{'Depth':>7}{'Toffoli depth':>15}")
    for n in WIDTHS:
        d = hybrid_cla(n)
        r = resources(d, d.circuit)
        print(f"  {n:>5}{r['lines']:>7}{r['gates']:>7}{r['cost']:>7}{r['T']:>9}{r['anc']:>9}{r['garbage']:>9}"
              f"{r['depth']:>7}{r['tdepth']:>15}")
    unchanged = all(circuit_cost(simplify(hybrid_cla(n).circuit, list(hybrid_cla(n).inputs.values()),
                                          hybrid_cla(n).ancilla, hybrid_cla(n).required).circuit)
                    == circuit_cost(hybrid_cla(n).circuit) for n in WIDTHS)
    report("no rule or simplification step changes it", unchanged)
    print()
    print("  All designs after all steps. P = Pareto-optimal in (cost, Toffoli depth, ancilla);")
    print("  P+ = Pareto-optimal when T-count and garbage outputs are added.")
    designs = dict(DESIGNS)
    designs["hybrid"] = hybrid_cla
    for n in WIDTHS:
        rows = []
        for key, make in designs.items():
            d = make(n)
            s = simplify(d.circuit, list(d.inputs.values()), d.ancilla, d.required).circuit
            rows.append((key, resources(d, s)))

        def pareto(r, keys):
            return not any(all(o[k] <= r[k] for k in keys) and any(o[k] < r[k] for k in keys)
                           for _, o in rows if o is not r)
        print(f"  n = {n}")
        print(f"    {'Design':<13}{'Cost':>6}{'T-count':>9}{'Tof. depth':>12}{'Ancilla':>9}{'Garbage':>9}   Pareto")
        for key, r in rows:
            mark = ("P" if pareto(r, ("cost", "tdepth", "anc")) else "-") + \
                   (" P+" if pareto(r, ("cost", "T", "tdepth", "anc", "garbage")) else "")
            print(f"    {key:<13}{r['cost']:>6}{r['T']:>9}{r['tdepth']:>12}{r['anc']:>9}{r['garbage']:>9}   {mark}")
    print()


def main() -> int:
    start = time.time()
    print("Phase 11: ancilla and garbage rules, hybrid adder\n")
    part_a()
    part_b()
    part_c()
    part_d()
    part_e()
    print(f"Time: {time.time() - start:.1f} s")
    print("\nAll checks passed." if failures == 0 else f"\n{failures} check(s) FAILED.")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

## Step 3: Run every check

Part D downloads the RevLib files from GitHub, as in Phase 8.

In [ ]:
!python verify_phase11.py

## Step 4: Try the rules on one adder

Change `NAME` and `BITS`. Names: `cla4_blocks`, `tsg_rca`, `peres_rca`, `cuccaro`, `vbe`, `draper_qcla`.

In [ ]:
from nadders import DESIGNS, prove_adder
from optimizer import circuit_cost, optimize
from arules import ancilla_optimize
from simplify import simplify

NAME, BITS = "cla4_blocks", 8

d = DESIGNS[NAME](BITS)
args = (d.circuit, list(d.inputs.values()), d.ancilla, d.required)
local = ancilla_optimize(*args)
print(d.name)
print("  quantum cost as built:              ", circuit_cost(d.circuit))
print("  RC rules only:                      ", circuit_cost(optimize(d.circuit).circuit))
print("  RC + rules A1-A5 (local):           ", circuit_cost(local.circuit))
print("  global simplifier (Phase 5):        ", circuit_cost(simplify(*args).circuit))
print("  rules used:", dict(local.counts))
print("  local result proved correct for every input:", prove_adder(d.with_circuit(local.circuit)))

## Step 5: Draw the hybrid adder

The 8-bit hybrid adder: eight PGA blocks (Peres gates), the carry-input gate, the P, G and C rounds of the prefix network, and the sum gates.

In [ ]:
import figures
from IPython.display import Image, display
from hybrid import hybrid_bands, hybrid_cla

figures.design_figure("fig_hybrid.png", hybrid_cla(8), bands=hybrid_bands(8))
display(Image(filename="fig_hybrid.png", width=950))